# 🎙️ beydahsaglam.com — 18 Blog Yazısını Tek Tıkla Toplu Seslendirme

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beydah/voice-clone_f5-tts_tr/blob/main/colab/Batch_Blog_Narration_Colab_TR.ipynb)
[![Website](https://img.shields.io/badge/Website-beydahsaglam.com-blue?style=flat&logo=googlechrome)](https://beydahsaglam.com)
[![GitHub](https://img.shields.io/badge/GitHub-beydah-181717?style=flat&logo=github)](https://github.com/beydah)

Bu özel Colab çalışma alanı, **F5-TTS Türkçe** sıfır örnekli (zero-shot) ses klonlama modeli ile **beydahsaglam.com** üzerindeki **18 yayınlanmış blog yazısının tamamını** tek bir seferde seslendirir.

### ⚡ Nasıl Çalışır?
1. **Tek Referans Ses:** 10 saniyelik referans ses kaydınızı yüklersiniz.
2. **18 Blog Otomatik Hazır:** Tüm blogların başlık, özet ve gövde metinleri bu deftere önceden gömülmüştür.
3. **Sıralı Toplu Üretim:** Model her yazıyı bağımsız cümlelere bölerek, fonetik düzeltmelerle stüdyo kalitesinde `<slug>-tr.wav` üretir.
4. **Tek Tıkla Zip İndirme:** Tüm ses dosyaları tek bir zip arşivi (`beydahsaglam_blog_sesleri_18.zip`) olarak otomatik bilgisayarınıza iner!

In [ ]:
#@title ⚙️ 0. Adım: GPU Kontrolü ve Gerekli Kütüphanelerin Kurulumu
#@markdown T4 GPU kontrol edilir, F5-TTS, Vocos vocoder ve ses işleme araçları kurulur (~40 sn):

import torch

# 1. GPU Kontrolü (Fail-Fast: CPU'da saatlerce beklemenizi önler)
if not torch.cuda.is_available():
    print("\n" + "!" * 72)
    print("⚠️  DİKKAT: GPU (CUDA) AKTİF DEĞİL! Şu anda CPU ortamındasınız.")
    print("    F5-TTS modeli CPU üzerinde cümle başına ~8-10 dakika sürer.")
    print("    GPU (T4) üzerinde ise cümle başına yalnızca 2-3 saniye sürer!")
    print("\n👉  Hemen GPU'ya geçmek için:")
    print("    1. Yukarıdaki Colab menüsünden: 'Çalışma Zamanı' (Runtime)")
    print("    2. 'Çalışma zamanı türünü değiştir' (Change runtime type)")
    print("    3. Donanım Hızlandırıcı (Hardware accelerator): 'T4 GPU' seçin")
    print("    4. 'Kaydet' butonuna basıp bu hücreyi tekrar çalıştırın.")
    print("!" * 72 + "\n")
    raise SystemExit("Lütfen T4 GPU'ya geçip bu hücreyi tekrar çalıştırın.")

print(f"🚀 GPU Aktif ve Hazır: {torch.cuda.get_device_name(0)} (Cihaz: CUDA)")

import subprocess

print("⏳ Gerekli paketler kuruluyor (f5-tts, pydub, soundfile, scipy)...")
subprocess.run(["pip", "install", "-q", "f5-tts", "pydub", "soundfile", "scipy"], check=True)

import transformers
transformers.pipeline = lambda *args, **kwargs: None

print("✅ Kurulum tamamlandı! 1. Adıma geçebilirsiniz.\n")


---
## 🎯 1. Adım: 10 Saniyelik Referans Sesinizi Yükleyin

Model ses tonunuzu ve diksiyonunuzu tek bir kısa ses kaydından öğrenecektir.

> **Okunacak 29 Harfli Altın Cümle:**
> *"Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir."*

In [ ]:
#@title 📤 1. Adım: Referans Sesi Yükleyin
#@markdown Çalıştırın ve referans sesinizi (.wav, .mp3, .m4a) seçin:

#@markdown ---
#@markdown 🎛️ **İsteğe Bağlı Ses İyileştirme Ayarı:**
otomatik_iyilestirme_uygula = False #@param {type:"boolean"}
#@markdown *(⚠️ Zaten Adobe Podcast / Audacity veya stüdyoda temizlenmiş bir ses yüklüyorsanız kutucuğu **İŞARETSİZ (False)** bırakın. Yalnızca gürültülü veya dip uğultusu olan ham kayıtlar için True yapın).*

import os
import soundfile as sf
import numpy as np
from scipy import signal
from pydub import AudioSegment
from IPython.display import Audio, display
from google.colab import files

print("🎙️ Lütfen 8-12 saniyelik referans sesinizi seçin:")
yuklenen = files.upload()
if not yuklenen:
    raise ValueError("⚠️ Lütfen bir ses dosyası yükleyin!")

ham_ses = list(yuklenen.keys())[0]
temiz_ref = "temiz_referans.wav"

# Her formatı standart 48kHz mono WAV formatına dönüştür
seg = AudioSegment.from_file(ham_ses).set_channels(1).set_frame_rate(48000)

if otomatik_iyilestirme_uygula:
    print("🎛️ Opsiyonel filtreler uygulanıyor (75Hz dip filtresi, sessizlik kırpma ve normalizasyon)...")
    temp_raw = "temp_raw_filter.wav"
    seg.export(temp_raw, format="wav")
    data, sr = sf.read(temp_raw)
    if len(data.shape) > 1:
        data = data.mean(axis=1)

    sos = signal.butter(4, 75, 'hp', fs=sr, output='sos')
    filtered = signal.sosfilt(sos, data)

    frame_len = int(sr * 0.03)
    energy = np.array([np.sum(filtered[i:i+frame_len]**2) for i in range(0, len(filtered)-frame_len, frame_len)])
    thresh = np.percentile(energy, 65) * 0.08
    active = np.where(energy > thresh)[0]
    trimmed = filtered[max(0, active[0]*frame_len - int(sr*0.08)) : min(len(filtered), (active[-1]+1)*frame_len + int(sr*0.12))] if len(active) > 0 else filtered

    peak = np.max(np.abs(trimmed))
    if peak > 0:
        trimmed = trimmed / peak * 0.89

    sf.write(temiz_ref, trimmed, sr)
    print(f"✨ Filtrelenmiş referans ses hazır ({len(trimmed)/sr:.1f} sn):")
    display(Audio(temiz_ref, rate=sr))
else:
    seg.export(temiz_ref, format="wav")
    data, sr = sf.read(temiz_ref)
    print(f"✅ Orijinal ses doğrudan korundu (filtresiz, stüdyo tonu korundu, {len(data)/sr:.1f} sn):")
    display(Audio(temiz_ref, rate=sr))


---
## 📚 2. Adım: Blog Korpusunu Yükleyin ve Ayarları Seçin

Bu hücre, beydahsaglam.com'daki 18 blog yazısının metinlerini hafızaya yükler.

In [ ]:
#@title 📚 2. Adım: Blog Korpusunu Yükleyin
#@markdown Seslendirme parametrelerini belirleyin:

referans_metni = "Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir."
konusma_hizi = 1.05 #@param {type:"slider", min:0.9, max:1.2, step:0.02}
sadakat_cfg = 2.2 #@param {type:"slider", min:1.5, max:3.0, step:0.1}
kalite_adim_sayisi = 32 #@param {type:"slider", min:16, max:32, step:4}
kac_yazi = "Tümü (18 Blog)" #@param ["Tümü (18 Blog)", "İlk 3 Test Blogu", "İlk 5 Blog"]

import json

# Gömülü 18 blog korpusu
BLOG_CORPUS = [
  {
    "index": 1,
    "slug": "n8n-zamanlama-saat-dilimi-deneyi",
    "slug_tr": "n8n-zamanlama-saat-dilimi-deneyi",
    "title_tr": "Saat 09.00 nerede? n8n zamanlamasında saat dilimini görünür yap",
    "summary_tr": "Aynı an İstanbul ve New York’ta farklı görünür; bazı yerel saatler sonbaharda iki kez yaşanır. Akışın saat dilimini, çalıştırma anını ve gösterim biçimini ayırarak zamanlamayı incele.",
    "word_count": 740,
    "estimated_duration_sec": 317,
    "narration_text": "Saat 09.00 nerede? n8n zamanlamasında saat dilimini görünür yap. Aynı an İstanbul ve New York’ta farklı görünür; bazı yerel saatler sonbaharda iki kez yaşanır. Akışın saat dilimini, çalıştırma anını ve gösterim biçimini ayırarak zamanlamayı incele.. “Her sabah saat dokuzda özeti hazırla” ifadesi insan için anlaşılır görünür. Bir otomasyonda ise hangi bölgenin dokuzu olduğu, tekrarın ne anlama geldiği ve çalışmanın hangi kayıtla doğrulanacağı açık olmalıdır. Bu yazı n8n zamanlamasını düşünmek için küçük bir çerçeve sunar. Deney alanı gerçek akışa bağlanmaz; sabit anların farklı saat dilimlerinde nasıl göründüğünü gösterir. An ile duvar saatini ayır Bir zaman çizgisindeki belirli an, farklı bölgelerde farklı saatlerle ifade edilir. UTC ile işaretlenmiş bir anı İstanbul ve New York için biçimlendirdiğinde farklı gösterimler elde edersin; olayın kendisi değişmez. “Her gün 09.00” ise tek bir an değil, yerel takvim üzerinde tekrar eden bir niyettir. Bu iki ihtiyacı aynı alanla ifade etmeye çalışmak belirsizliği büyütebilir. Önce yaptığın işin türünü yaz. Tek seferlik bir gönderim mi, her iş günü rapor mu, yoksa başka sistemden gelen zaman damgasının gösterimi mi? Ardından gerekli bölge bilgisini belirle. Bir kullanıcı arayüzünde saatin yerelleştirilmesi, arka plandaki tetikleyicinin aynı bölgeye göre çalıştığını göstermez. Görünen saat ile çalışan kuralı ayrı kontrol et. Akışın kullandığı bölgeyi bul n8n’in Schedule Trigger belgeleri, akış saat dilimi ayarlanmışsa bu değerin, yoksa kurulumun saat diliminin kullanıldığını açıklar. Bu yüzden yalnızca bilgisayar saatine bakmak yeterli değildir. Akış ayarını ve kurulum ayarını birlikte incele; beklediğin davranışı hangisinin belirlediğini not et. Kendi barındırdığın ortamda ilgili çevre değişkeni yapılandırması da bu incelemenin parçası olabilir. Kurulum değerini değiştirmek, bir akış için özel değer tanımlamakla aynı kapsamda değildir. Önce hangi akışların etkileneceğini belirle. Belgeler zamanlanmış akışın yayımlanması gerektiğini de anlatır; düzenleyicide ayarı değiştirmek gerçek zamanlanmış çalışmanın başladığına tek başına kanıt değildir. Bu adımları ekrandaki durum ve yürütme kaydıyla ilişkilendir. Bölge adı ile UTC farkını karıştırma Europe/Istanbul veya America/New York gibi IANA bölge adları ile belirli andaki UTC farkı farklı bilgilerdir. Deneyde İstanbul aynı örneklerde üç saat ileride görünürken New York’un kış ve yaz gösterimi farklılaşır. Buradan bütün ülkeler için kalıcı bir fark tablosu çıkarmaya çalışma. Bölge kuralları ve tarih, gösterimin birlikte değerlendirilmesi gereken parçalarıdır. Tarayıcıdaki gösterim Intl.DateTimeFormat üzerinden yapılır. Bu, belirtilen bölgeye göre bir anı biçimlendirmeye yarar; kendi başına bir takvim tetikleyicisi değildir. Kullanılan ortamın zaman dilimi verileri de gerçek uygulamada önemlidir. Bir istemcide doğru görünen örneği, sunucudaki her sürüm ve çalışma koşulu için kapsamlı doğrulama olarak sunma. Geri alınan saati ayrı bir örnek yap Sonbahardaki saat geri alma geçişi ilginç bir sınırdır. Deneyde New York için 1 Kasım 2026 tarihindeki iki ayrı UTC anını seç. İkisi de yerel 01.30 olarak görünür, ancak biri GMT-04.00, diğeri GMT-05.00 farkını taşır. Yalnızca saat ve dakikayı kaydetmek bu iki olayı birbirinden ayırmaya yetmez. Tam an ve bölge bağlamı birlikte anlam sağlar. Bu örnek, gerçek bir zamanlayıcının belirsiz yerel saati nasıl ele aldığını kanıtlamaz. Kullandığın ürünün kuralını belgelerden ve kontrollü testten öğrenmelisin. İlkbaharda bulunmayan bir yerel saat de ayrı bir sınır örneğidir. Bu deney o durumu hesaplamıyor; kapsamı genişletmeden önce neyin gösterildiğini açık tutmak daha doğru bir açıklamadır. Beklentiyi gerçek kayda bağla Bir test için beklenen yerel saati, bölgeyi ve karşılık gelen anı aynı notta tut. Sonra yürütme kaydını açıp gerçekten başlayan işlemi karşılaştır. Bildirimin geliş saati tek başına tetikleme saati değildir; kuyruk, işlem süresi ve bildirim yolu araya girebilir. “Zamanında geldi” değerlendirmesini hangi olaya göre yaptığını belirtmek tartışmayı somutlaştırır. PDF çıktısı seçilen sabit anı, bölgeyi ve gösterimi taşır. Bu belgeyi gerçek çalıştırma kaydının yerine değil, yanına koy. İlk turda tek bir akış için beklenti ve gözlenen sonucu eşleştirmek yeterli olabilir. Sonraki turda yaz saati sınırlarını, çalışma ortamı farklılıklarını ve tekrar kuralını kontrollü biçimde genişlet. Böylece saat dilimi bir varsayım olmaktan çıkıp incelenebilir bir karar haline gelir. Bir çalıştırma notu örneği Sabah raporu için beklenen olayı “raporun hazırlanması” diye yazdıysan, e-postanın gelen kutusuna düşmesini aynı olay gibi değerlendirme. Test notunda tetikleme, işlem başlangıcı, tamamlanma ve bildirim zamanlarını ayrı satırlarda tutabilirsin. Böylece yanlış saat dilimi ile uzun işlem süresi birbirine karışmaz. Bu ayrım, bir gecikmenin hangi bölümde oluştuğunu araştırmak için de yararlıdır. Bir sonraki denemede yalnızca bir ayarı değiştir. Akış bölgesini değiştirirken aynı anda tekrar kuralını ve bildirim servisini değiştirirsen sonucun neden farklılaştığını ayırmak zorlaşır. Denenen ayarların kısa bir kopyası ve yürütme kaydına ait referans yeterli olabilir. Kamuya açık bir yazıda gerçek alıcı veya özel hesap bilgisini paylaşman gerekmez. Beklenen ile gözlenen zamanı aynı gösterim biçimine getirdikten sonra farkı yorumla; yalnızca ekrandaki saatlerin farklı görünmesini hata sayma. Kaynaklar - n8n — Schedule Trigger - n8n — Timezone and localization - MDN — Intl.DateTimeFormat"
  },
  {
    "index": 2,
    "slug": "ai-arayuzunde-guvenli-kismi-guncelleme",
    "slug_tr": "ai-arayuzunde-guvenli-kismi-guncelleme",
    "title_tr": "AI arayüzünde küçük güncelleme: bütün ekranı modele bırakma",
    "summary_tr": "Model çıktısını çalıştırılacak arayüz kodu yerine sınırlı bir veri değişikliği olarak ele al. Eski yanıtı, bilinmeyen alanı ve geçersiz değeri reddeden küçük bir JSON deneyini dene.",
    "word_count": 813,
    "estimated_duration_sec": 348,
    "narration_text": "AI arayüzünde küçük güncelleme: bütün ekranı modele bırakma. Model çıktısını çalıştırılacak arayüz kodu yerine sınırlı bir veri değişikliği olarak ele al. Eski yanıtı, bilinmeyen alanı ve geçersiz değeri reddeden küçük bir JSON deneyini dene.. Bir AI asistanı kartın başlığını değiştirecek, özeti kısaltacak ve ilerleme göstergesini güncelleyecek olsun. Bütün arayüzü yeniden üretmek yerine yalnızca bu üç alana yönelik küçük bir veri değişikliği istemek mümkün. Buradaki öneri belirli bir React özelliği değil; model çıktısını uygulamanın kontrol ettiği bir sözleşmeye dönüştüren tasarım yaklaşımıdır. Deney gerçek model çağrısı yapmaz ve ziyaretçinin yazdığı metni kod olarak çalıştırmaz. Önce değişebilen alanları belirle Ekrandaki her alanı modele açmak zorunda değilsin. Bu örnekte title, summary ve progress değişebilir; request alanı yanıtın hangi işe ait olduğunu belirtir. Bilinmeyen alanlar kabul edilmez. Böylece modelin yanlışlıkla başka bir düğmenin hedefini, kullanıcı kimliğini veya görünmeyen ayarı değiştirmesi bu veri yolunun parçası olmaz. Alan listesi küçük kaldığında beklentiyi test etmek de kolaylaşır. Sözleşmeyi örnek bir başarılı yanıtla sınırlama. Başlık boşsa, özet çok uzunsa veya ilerleme yüzün üzerindeyse ne olacağını yaz. İstemcide bir sayıyı kaydırıcıyla sınırlamak tek başına yeterli değildir; çıktı başka bir kaynaktan geldiğinde aynı kurallar uygulanmalı. Gerçek uygulamada sunucunun kabul ettiği değişiklikler de kendi yetki ve doğrulama sınırlarını taşımalıdır. Ayrıştırma ile kabul kararını ayır Bir metin JSON olarak ayrıştırılabiliyorsa yalnızca biçim kontrolünün bir bölümü geçilmiştir. Bir dizi, yanlış türde bir değer veya beklenmeyen alan da geçerli JSON olabilir. Uygulamanın kabul kararı, kendi şemasına ve iş kurallarına dayanmalıdır. Deneyde sayı beklenen yere metin vermeyi, boş bir başlık göndermeyi ve bilinmeyen alan eklemeyi deneyebilirsin. Ret durumunda anlaşılır bir sonuç üretmek önemlidir. Ekranı boşaltmak veya yarım uygulanmış bir değişiklik bırakmak yerine son geçerli durum korunabilir. Kullanıcı neyin kabul edilmediğini görürken önceki metni kaybetmez. Bu davranış hata gizlemek değildir; başarısız öneriyle son geçerli ekranın birbirinden ayrılmasıdır. Hata kaydı tutulacaksa modelin bütün çıktısını gereksiz yere günlüklemek de varsayılan olmamalı. Yanıtı güncel işle eşleştir Asenkron yanıtlar başlama sırasıyla bitmek zorunda değildir. Kullanıcı ikinci bir istek gönderdiğinde ilk istek hâlâ çalışıyor olabilir. İlk yanıtın geç gelmesi, güncel kartı eski bir sonuçla değiştirmemeli. Deneyde güncel istek kimliği başlangıçta ikidir ve 1–10 arasında değiştirilebilir; request değeri seçilen güncel kimlikle eşleşmeyen parça reddedilir. Bu sayı gerçek kullanıcı veya kayıt kimliği değildir, yalnızca sıralama ilişkisini açıklayan bir araçtır. Gerçek uygulamada yeni iş başlayınca sürümün nasıl arttığını, iptal edilen isteğin ne olduğunu ve bileşen kapandığında sonucun nereye gideceğini tanımla. İstek kimliği kontrolüyle ağ isteğini iptal etme aynı iş değildir. Birini uygulamak diğerinin etkisini otomatik sağlamaz. Kullanıcıya eski yanıtın uygulanmadığını göstermek, sessizce karışan sonuçlardan daha anlaşılır bir davranış sunabilir. Mevcut durumu bilinçli koru Kısmi değişiklik bütün durumun yerine geçmek zorunda değildir. Bu deney her JSON parçasını sabit başlangıç kartından hesaplar: Original title, Existing summary ve 0. Yalnızca progress gönderilirse başlık ve özet bu başlangıç değerlerinden alınır; önceki kabul edilen parçalar biriktirilmez. Gerçek bir uygulamada son geçerli duruma birleştirme davranışı ayrıca uygulanmalıdır. React’in nesne durumu rehberi, durumdaki nesneleri doğrudan değiştirmek yerine yeni nesnelerle güncellemeyi açıklar. İç içe yapıda ilgili katmanları kopyalamak gerekir; deney düz bir nesneyle sınırlıdır. Önizlemede kabul edilen alanları göstermek incelemeyi kolaylaştırır. Bir değişiklik başlığı mı etkiledi, yalnızca ilerlemeyi mi? Kullanıcı bu ayrımı ekrandan anlayabilsin. Kaydetme veya yayımlama gibi dış etkileri aynı işlem içinde gizleme. Bir görsel önizlemenin değişmesi, veritabanındaki kaydın güncellendiğini veya yayına çıkıldığını kanıtlamaz; her adım kendi sonucunu bildirmelidir. Küçük ret örnekleriyle başla Denemeye geçerli bir progress değişikliğiyle başla. Sonra aynı isteğe bilinmeyen bir alan ekle, request değerini eskisine çevir ve aralık dışı bir sayı gönder. Her ret sonrasında deneyin sabit başlangıç kartına döndüğünü gözle; önceki kabul edilen önizleme saklanmaz. Gerçek uygulamada son geçerli ekranı korumak istiyorsan bu davranışı ayrıca uygula ve test et. Bu sıra, yalnızca mutlu yolun çalıştığı bir demodan daha fazla bilgi verir. Serbest metni HTML olarak yorumlayan bir kestirme ekleme. PDF çıktısı seçtiğin JSON metnini ve modelin hesapladığı kabul sonucunu içerir. Gerçek sır veya kişisel veri yazma; bu alan yalnızca sahte örnekler içindir. Çıktıyı ekip içi bir sözleşme notuna ekleyebilirsin. Sonraki adımda tek bir gerçek ekranın alanlarını ve ret davranışını tanımlamak, tüm uygulamayı dinamik üreteceğini iddia etmekten daha gözlenebilir bir geliştirme hedefidir. Kabul kaydında üç sonucu ayır Bir değişiklik önerisi için ayrıştırıldı, kabul edildi ve ekrana uygulandı sonuçlarını ayrı düşün. Sözdizimi doğru bir metin, beklenmeyen alan nedeniyle reddedilebilir. Kabul edilen bir değişiklik de bileşen kapandığı için artık gösterilmeyebilir. Bu durumları tek bir “tamamlandı” işaretine sıkıştırmak hata araştırmasını zorlaştırır. Deney yalnızca kabul modelini gösterir; gerçek uygulamada sonraki adımlar ayrıca gözlenmelidir. Kısa bir test kaydında giriş örneğini, beklenen kararı ve kalan ekranı birlikte tut. Başlık güncellenirken özetin korunması beklentisi açık olsun. Ardından aynı öneriyi eski istek kimliğiyle tekrarla ve parçanın reddedildiğini doğrula. Bu deneyde ret sabit başlangıç kartını gösterir; gerçek uygulamada son geçerli durumu koruma kuralını ayrı sınamalısın. Kopyalanan test metninde gerçek kullanıcı bilgisi bulunmasın. Böyle bir kayıt, sonraki şema değişikliğinde neyin kasıtlı olarak değiştiğini karşılaştırmayı kolaylaştırır. Yeni bir alan eklemek yalnızca arayüz kontrolü eklemek değildir; kabul ve ret örnekleri de o yeni sınırı kapsamalıdır. Kaynak - React — Updating objects in state"
  },
  {
    "index": 3,
    "slug": "turkce-ingilizce-yazi-url-eslestirme",
    "slug_tr": "turkce-ingilizce-yazi-url-eslestirme",
    "title_tr": "İki dil, iki adres: Türkçe ve İngilizce yazıyı doğru eşleştir",
    "summary_tr": "Türkçe bir yazının İngilizce adresi aynı kelimeleri taşımak zorunda değil. Çeviri hazır olduğunda dil çiftini kur; hazır değilken boş bir İngilizce sayfa üretme. Küçük deneyde eksik eşleşmeleri bul.",
    "word_count": 728,
    "estimated_duration_sec": 312,
    "narration_text": "İki dil, iki adres: Türkçe ve İngilizce yazıyı doğru eşleştir. Türkçe bir yazının İngilizce adresi aynı kelimeleri taşımak zorunda değil. Çeviri hazır olduğunda dil çiftini kur; hazır değilken boş bir İngilizce sayfa üretme. Küçük deneyde eksik eşleşmeleri bul.. Bir yazıyı İngilizceye çevirmek yalnızca ikinci bir metin alanını doldurmak değildir. Okuyucu yazıya listeden gelir, bir bağlantıyı paylaşır, dil değiştirir veya eski adresi açar. Bu yolların aynı içeriğin doğru sürümüne ulaşması gerekir. Bu sitede Türkçe ve İngilizce metinler ayrı adreslere sahip olabilir; henüz çevirisi olmayan içerik İngilizce listede görünmez. Aşağıdaki yaklaşım, bu davranışın nasıl gözden geçirilebileceğini anlatır. Adresi çevirinin kimliğiyle karıştırma Türkçe başlık için seçilen slug, İngilizce okuyucunun anlayacağı en iyi ifade olmayabilir. İki ayrı alan kullanmak bu seçimi kolaylaştırır. Ancak iki adresin aynı içeriğe ait olduğunu yalnızca metin benzerliğinden çıkarmak kırılgandır. Arka planda aynı içerik kaydının dil sürümleri olarak ilişkilendirilmeleri, başlık veya adres değiştiğinde eşleşmeyi korumaya yardımcı olur. Bir kayıt için Türkçe adresi ve İngilizce adresi yan yana yaz. Ardından hangi dilin gerçekten yayıma hazır olduğunu belirt. Slug alanının dolu olması, çevirinin tamamlandığını kanıtlamaz. Başlık, özet, ana metin ve gerekli materyallerin o dilde bulunmasını ayrıca kontrol et. Bu ayrım, bir adres planlamak ile okuyucuya kullanılabilir bir sayfa sunmak arasındaki farkı görünür kılar. Olmayan çeviriyi varmış gibi sunma Bir Türkçe yazının henüz İngilizce sürümü yoksa İngilizce listede aynı Türkçe metni göstermek okuyucunun beklentisini bozar. Bu sitede eksik sürüm listeye ve sitemap’e eklenmez. Yönetici panelinde çeviri hazırlanabilir; kamuya açık bir alternatifin varlığı ise yayın durumuyla belirlenir. Böylece taslak hazırlama süreci ziyaretçinin karşılaştığı dil karışıklığına dönüşmez. Dil değiştirici için de bilinçli davranış gerekir. Mevcut yazının başka dilde karşılığı yoksa karşılık varmış gibi bir yazı bağlantısı üretme. Kullanıcıyı genel blog listesine götürüyorsan bunun aynı yazının çevirisi olmadığını açıkça belirt. Başarılı yönlendirme yalnızca HTTP 200 almak değildir; okuyucu doğru beklentiyle doğru içeriğe ulaşmalıdır. Dil alternatiflerini karşılıklı kur Google’ın yerelleştirilmiş sürümler rehberi, hreflang işaretlerinin tam adresler kullanmasını ve sürümlerin birbirini karşılıklı göstermesini açıklar. Her sayfa kendi sürümünü de belirtilen kümede taşır. Bir tarafta İngilizce alternatif varken İngilizce sayfada Türkçe dönüşünün eksik olması, kontrol edilmesi gereken bir eşleşme problemidir. Farklı yöntemleri gereksiz yere çoğaltmak yerine tutarlı bir yöntem seçmek incelemeyi kolaylaştırır. Bu teknik işaret, içerik kalitesinin yerine geçmez. Sayfa başlığı, görünür dil, bağlantı etiketi ve metin gerçekten aynı sürümü anlatmalı. Kaynak koddaki doğru etiket ile ekrandaki yanlış İngilizce başlık birlikte bulunabilir. Bu nedenle yalnızca etiket araması yapmak yerine sayfanın okuyucuya sunduğu deneyimi de açıp gör. Canonical başka bir soruya cevap verir Canonical, tercih edilen adresle ilgili bir işarettir; dil alternatifleri ise sürümler arasındaki dil ilişkisini anlatır. Bu sitedeki gerçek TR/EN sürümlerinde her sayfanın kendi yerelleştirilmiş adresi canonical olarak kullanılır. İngilizce sayfayı otomatik olarak Türkçe adrese işaretlemek, kurmaya çalıştığın iki ayrı sürüm davranışıyla çelişebilir. Her uygulamanın bağlamı ayrıca değerlendirilmelidir. Kontrol tablosunda sayfanın açılan adresi, canonical adresi ve dil alternatiflerini ayrı sütunlara yaz. Eski slug’dan gelen yönlendirme de nihai adresle tutarlı olsun. Bir araçta yeşil işaret görmekten önce hangi ilişkinin doğrulandığını bilmek gerekir. Örneğin canonical bulunması, bağlantının doğru sürüme gittiğini tek başına kanıtlamaz. Eski bağlantıları eşdeğer içerikle karşıla Adres değişince daha önce paylaşılmış bağlantıları düşün. Aynı yazının yeni Türkçe adresine giden eski bağlantı için kalıcı yönlendirme anlamlı olabilir. Ancak karşılığı bulunmayan her eski WordPress adresini ana sayfaya göndermek okuyucunun niyetini korumaz. Önce gerçek eşdeğeri belirle; taslak veya kaldırılmış içeriğin durumunu farklı değerlendir. Bu çalışma kör bir toplu yönlendirme listesi değildir. Deney alanında İngilizce sürümün varlığını, karşılıklı eşleşmeyi ve canonical tutarlılığını değiştirebilirsin. Model gerçek siteyi taramaz; üç kararın nasıl farklı sorunlara yol açtığını gösterir. Sonucu PDF’ye aktararak kendi kontrol tablonun yanına koy. İlk turda tek bir yazıyı listeden eski bağlantıya kadar doğrulamak, bütün site için kanıtsız bir SEO iddiasından daha yararlı bir başlangıçtır. Okuyucunun yolunu da dene Örnek bir testte Türkçe blog listesinden yazıyı aç, adresi kopyala ve dil değiştiriciyi kullan. İngilizce başlığın ve metnin aynı konuyu anlattığını kontrol et. Sonra İngilizce adrese doğrudan yeni bir sekmeden git. Bu iki yol farklı sonuç veriyorsa yalnızca liste bağlantısını düzeltmek yeterli olmayabilir. Tarayıcıdaki geçici durumun hatayı gizleyip gizlemediğini de değerlendirmiş olursun. Aynı kaydın çevirisi taslağa döndüğünde beklenen davranışı ayrıca yaz. İngilizce liste, sitemap ve alternatif etiketler birbiriyle tutarlı mı? Türkçe sürüm kullanılmaya devam ediyor mu? Her iki dilin aynı anda yayımlanmasını zorunlu kılmadan bu ilişkileri korumak mümkündür. Kontrol kaydında hangi adresin hangi yayın durumunda denendiği bulunsun. Bir dilin geçici olarak eksik olması ile yanlış içeriğe yönlendirme yapılması ayrı bulgulardır; çözümü de ayrı olabilir. Kaynak - Google Search Central — Localized versions"
  },
  {
    "index": 4,
    "slug": "n8n-akisini-paylasmadan-once-gizli-bilgi-kontrolu",
    "slug_tr": "n8n-akisini-paylasmadan-once-gizli-bilgi-kontrolu",
    "title_tr": "n8n akışını paylaşmadan önce: anahtar nerede saklanıyor?",
    "summary_tr": "Bir workflow dosyası yalnızca düğümlerden oluşmaz. Kimlik bilgisi adları, cURL başlıkları ve örnek veriler de paylaşılabilir. Dört küçük senaryoyla dosyayı paylaşmadan önce neyi kontrol edeceğini gör.",
    "word_count": 763,
    "estimated_duration_sec": 327,
    "narration_text": "n8n akışını paylaşmadan önce: anahtar nerede saklanıyor?. Bir workflow dosyası yalnızca düğümlerden oluşmaz. Kimlik bilgisi adları, cURL başlıkları ve örnek veriler de paylaşılabilir. Dört küçük senaryoyla dosyayı paylaşmadan önce neyi kontrol edeceğini gör.. Bir otomasyonu anlatmanın en hızlı yolu çoğu zaman workflow JSON dosyasını göndermektir. Alıcı düğümleri görür, örneği içe aktarır ve kendi hesabına bağlar. Ancak dosyanın çalışıyor olması, paylaşılmaya uygun olduğunu göstermez. Bu yazı kişisel n8n örneklerinde uygulanabilecek bir gözden geçirme yaklaşımıdır. Amaç bütün dışa aktarımları tehlikeli ilan etmek değil, veri ile yetki arasındaki sınırı görünür kılmaktır. Referans ile anahtar aynı şey değildir Bir credential kaydına verilen kimlik, ilgili sırrın kendisi olmak zorunda değildir. Yine de bu ayrım dosyayı otomatik olarak herkese açık yapmaz. n8n’in dışa aktarma belgeleri, JSON içinde credential adları ve kimliklerinin bulunabileceğini açıklar. Adlandırmanın müşteriyi veya ortamı ele verdiği bir düzende bu alanlar bile gereksiz bilgi taşıyabilir. Paylaşım kopyasında alanın hangi görevi gördüğünü anlayarak karar ver. Örneğin “Müşteri A canlı ödeme hesabı” yerine “demo-http-credential” kullanmak, örneğin mantığını anlatırken gerçek ilişkiyi gizler. Ancak yalnızca adı değiştirmek gerçek bağlantıyı veya gömülü başlığı kaldırmaz. Her alanı işlevine göre incelemek gerekir. Bir credential referansının görünmesi ile doğrudan erişim sağlayan değerin görünmesi farklı bulgulardır; ikisini aynı etiket altında kaybetme. cURL ile gelen başlıkları ayrıca kontrol et Bir API örneğini cURL komutundan içe aktardıysan HTTP Request düğümündeki başlıkları gözden geçir. n8n belgeleri bu yolla alınan düğümlerin kimlik doğrulama başlıkları içerebileceğini özellikle belirtir. Authorization alanı, özel API başlıkları veya gövde içindeki token benzeri değerler beklediğinden farklı yerde kalmış olabilir. “Credential kullandım” demek bütün düğümlerin aynı yöntemi kullandığını kanıtlamaz. Önce gerekli alanların listesini çıkar: yöntem, adresin genel biçimi, örnek gövde ve beklenen yanıt. Sonra gerçek erişim değerlerini paylaşım kopyasından kaldır. Alıcıya kendi credential kaydını nerede bağlayacağını anlat. Gerçek anahtarı açıklama metnine taşımak, düğümden kaldırmanın amacını bozar. Örnek komutları ve hata ekranlarını da aynı kapsamda değerlendir. URL ve günlük yeni bir kopya üretir Anahtarın istekte kullanılması ile URL sorgusuna yazılması farklı paylaşım yolları oluşturur. Adresler tarayıcı geçmişi, sunucu kayıtları ve kopyalanan bağlantılar gibi alanlara taşınabilir. Günlükler de hata ayıklamak için yararlı olsa bile sonsuza kadar özel kalacağı varsayılan bir kasa değildir. Gereksiz gizli değeri kaydetmemek, sonradan her kopyasını bulmaya çalışmaktan daha yönetilebilir bir başlangıçtır. Deney alanında aynı sahte değeri başlık, günlük, URL ve referans olarak görebilirsin. Günlük maskelemesini açmak yalnızca o gösterimdeki metni değiştirir; URL’deki değeri temizlemez. “Güvenli” sonucu, bu dar öğretici modelde belirli bir paylaşım yolunun elendiğini anlatır. Gerçek bir Authorization başlığının her hedefe gönderilmesi güvenli değildir; hedefin doğruluğu, bağlantı ve erişim kapsamı ayrıca gerekir. Çıktı ve ekran görüntüsü de paketin parçasıdır Paylaşılacak şey yalnızca JSON olmayabilir. Örnek yanıt, sabitlenmiş veri, hata mesajı ve video kaydı da kişisel veya hesapla ilişkili bilgi içerebilir. Paylaşım paketini açıp alıcının göreceği bütün parçaları sırayla incele. Bir dosya temizlenirken açıklama ekranında eski değerin kalması sık karşılaşılabilecek bir tutarsızlıktır. Arama yapmak yardımcıdır ama bağlamı anlamanın yerine geçmez. Sahte veri kullanırken biçimi korumak yararlıdır. Gerçek e-posta yerine örnek alan adı, gerçek token yerine açıkça sahte bir gösterim kullan. Böylece alıcı hangi alanın değişeceğini anlar; örnek çalıştırıldığında gerçek kişiye mesaj gitmez. Beklenen sonucu ve dış etki oluşturmayan test yolunu birlikte yaz. Bir eğitim örneğinin başarısı gerçek müşteri kaydı üretmesine bağlı olmamalı. Temiz kopyayı yeniden içe aktar Son kontrolü, hazırladığın dosyayı temiz bir test ortamında açarak yap. Eksik credential bağlantısı beklenen yerde mi görünüyor? Örnek gövde anlaşılır mı? Hata çıktısı paylaşılabilir mi? Bu tur, dosyayı temizlerken öğretici kısmın da bozulup bozulmadığını gösterir. Tarih ve kapsam notu eklemek sonraki okuyucunun örneğin hangi koşullarda denendiğini anlamasını sağlar. Gerçek bir anahtar açığa çıktıysa bunu yalnızca içerik düzenleme işi olarak ele alma. Sağlayıcının iptal ve yenileme adımlarını uygula; eski değerin kopyalarını silmek tek başına erişimi sona erdirmez. Bu sayfanın PDF çıktısı da bir test notudur, güvenlik sertifikası değildir. Önce tek bir paylaşım kopyasını incelemek ve bulguları somut alanlarla kaydetmek iyi bir başlangıçtır. Paylaşım incelemesini kayıt altına al Örnek bir HTTP akışında dosya, açıklama ve ekran görüntüsünü üç ayrı satır olarak düşün. Dosyada credential referansı, açıklamada örnek komut, ekran görüntüsünde ise çıktı alanı bulunabilir. Her satır için gerçek hesapla bağlantı taşıyan alanı, yapılan değişikliği ve yeniden deneme sonucunu kaydet. Bir satırın temizlenmesi diğer iki satırı otomatik temizlemez. Bu küçük tablo incelemenin kapsamını görünür yapar. Temiz kopyayı hazırlayan kişi ile paylaşan kişi farklıysa son dosyanın hangisi olduğu da açık olmalı. Dosya adına tarih eklemek tek başına içeriğin değişmediğini kanıtlamaz; son paketi yeniden açıp gözden geçir. İlgisiz dosyaları aynı arşive koyma. Paylaşım sonrası yeni bir örnek yanıt eklenirse önceki incelemenin onu kapsadığını varsayma. Kısa bir değişiklik notu, kontrolün hangi parçalar için tekrarlandığını anlatabilir. Buradaki amaç uzun bir bürokrasi değil, gerçek bir paylaşım sınırını kaybetmemektir. Kaynak - n8n — Export and import workflows"
  },
  {
    "index": 5,
    "slug": "yapay-zeka-agent-yetki-karti",
    "slug_tr": "yapay-zeka-agent-yetki-karti",
    "title_tr": "Bir agent ne yapabilir? Yetkiyi görev metninden ayır",
    "summary_tr": "“Bu işi hallet” demek, bir araca bütün hesaplarda işlem yapma yetkisi vermez. Okuma, taslak hazırlama ve dışarı gönderme sınırlarını küçük bir yetki kartıyla görünür hale getir.",
    "word_count": 754,
    "estimated_duration_sec": 323,
    "narration_text": "Bir agent ne yapabilir? Yetkiyi görev metninden ayır. “Bu işi hallet” demek, bir araca bütün hesaplarda işlem yapma yetkisi vermez. Okuma, taslak hazırlama ve dışarı gönderme sınırlarını küçük bir yetki kartıyla görünür hale getir.. Bir bilgi asistanı düşün: üç kaynağı okuyabiliyor, bir özet hazırlıyor ve sonucu e-postayla gönderebiliyor. Kullanıcının “haftalık özeti hazırla” demesi, bu üç işlemin aynı yetkiyle yürütülmesini gerektirmez. Okuma hakkı, taslak yazma hakkı ve gerçek alıcıya gönderme hakkı farklı sınırlar olabilir. Bu yazı, bu sınırları kişisel bir otomasyon örneğiyle görünür kılan tasarım yaklaşımıdır; belirli bir ürünün güvenlik değerlendirmesi değildir. Görev ile yetkiyi ayrı satırlara yaz Görev metni beklenen sonucu anlatır. Yetki kartı ise hangi veriye hangi işlemin uygulanabileceğini açıklar. “Özeti hazırla” görevine karşılık, yalnızca seçilmiş kaynakları okuma ve yöneticiye açık bir taslak oluşturma yetkisi verilebilir. Gönderim bu karta dahil olmayabilir. Böylece aracın neyi yapabilecek durumda olduğu ile o anda neyi yapmasına izin verildiği birbirine karışmaz. Kartta kullanıcı, hedef kayıt, izin verilen işlem ve kapsamın geçerliliği bulunsun. “Bütün dosyalar” yerine ilgili klasör, “bütün alıcılar” yerine seçilmiş adres gibi somut bir sınır düşün. Bunlar evrensel bir izin şeması önerisi değil, kendi akışındaki belirsizliği yakalamak için sorulardır. Bir veri alanını neden açtığını açıklayamıyorsan önce daha dar kapsamla dene. Yetkisiz durumda varsayılan sonucu tanımla Bir işlem için açıkça izin bulunmuyorsa ne olacağını koddan önce yaz. Sessizce devam etmek, başka bir araçla aynı işi denemek veya yetkiyi tahmin etmek yerine anlaşılır bir ret sonucu üretilebilir. Bu sonuç kullanıcıyı çıkmaza sokmamalı: hangi kapsamın eksik olduğunu ve güvenli sonraki adımı açıklamalı. Ancak ret mesajı gereksiz özel veri veya başka hesapların varlığını da açığa çıkarmamalı. OWASP’ın yetkilendirme rehberi en az ayrıcalık, varsayılan ret ve her istekte izin denetimi gibi ilkeleri ele alır. Bunları kendi örneğine çevir: kaynak okuma izinli, taslak kaydı ayrı bir kapsamda, dış gönderim güncel onaya bağlı olsun. Bu ilkeler bir kütüphane adı seçmekle tamamlanmaz; davranışın gerçekten uygulandığını gözlemek gerekir. Onayı belirli bir dış etkiye bağla Gönderim için onay istendiğinde kişi hangi metni, hangi alıcıyı ve hangi işlemi onayladığını görebilsin. Sonra metin değişirse eski onayın yeni sürümü kapsayıp kapsamadığını sorgula. “Daha önce evet demişti” ifadesi yeterli bir işlem kaydı değildir. Aynı şekilde bir taslağın kaydedilmesi, onun yayımlanmasına veya başka bir hizmete gönderilmesine otomatik izin oluşturmaz. Bu ayrımı arayüzde de koru. Önizleme, taslak kaydı ve gönderim aynı etiketle sunulursa kullanıcı farklı dış etkileri ayırt etmekte zorlanır. Her kontrolün sonucu açık olsun. İptal veya ret durumunda metnin korunması da önemlidir; kişi çalışmasını kaybetmeden kapsamı yeniden gözden geçirebilmeli. Bir onay düğmesini kullanışlı yapan şey yalnızca belirgin rengi değildir. Denetimi doğru sınırda uygula Tarayıcıdaki bir düğmeyi gizlemek, doğrudan isteğin engellendiğini göstermez. Gerçek uygulamada sunucu da kullanıcıyı, kayıt sahipliğini ve izin verilen işlemi değerlendirmeli. İstemciden gelen “onaylı” alanını tek başına güvenilir sayma. Onay kaydının hangi iş sürümüne ait olduğu, hedefin değişip değişmediği ve kaydın hâlâ geçerli olup olmadığı birlikte incelenebilir olmalı. Karmaşık bir akışta aynı kapsamın farklı katmanlarda farklı yorumlanması mümkündür. Bunu küçük bir tabloyla yakala: arayüzün gösterdiği işlem, API’nin kabul ettiği işlem ve dış serviste oluşan etki. Üçü aynı işi mi anlatıyor? Gönderim reddedilirken taslak siliniyorsa veya başka hedefe yöneliyorsa sınırın yalnızca bir kısmı korunuyor olabilir. İzin verilmeyen örneği de çalıştır Test kümesinde yalnızca doğru kullanıcı ve doğru kayıt bulunmasın. Başka bir kayıt kimliği, süresi geçmiş onay ve onaydan sonra değişmiş alıcı için beklenen sonucu yaz. Kontrollü ortamda her durumda dış etkinin oluşmadığını bağımsız kayıtla kontrol et. Ekrandaki hata mesajı tek başına gönderim yapılmadığını kanıtlamaz. Tersine, zaman aşımı da işlemin kesin başarısız olduğunu göstermez. Deney alanında üç yetki kartını dört işlemle eşleştirebilirsin. Bu sade modelde kalıcı silme her zaman reddedilir; gerçek bir servise çağrı yapılmaz. Sonucu PDF indirerek kendi tasarım notlarınla karşılaştır. Bir sonraki gerçek uygulama turunda tek bir izinli ve tek bir reddedilen örnek seçmek, henüz gözlenmemiş geniş bir güvenlik iddiasından daha somut başlangıç sağlar. Küçük bir kabul kaydı örneği Haftalık özet için bir deneme kaydı oluşturduğunu düşün. İlk satırda kaynak okuma işlemi, ikinci satırda taslağın kaydı, üçüncü satırda dış gönderim yer alsın. Her satıra beklenen sonucu ve bağımsız kanıtı yaz. İlk ikisi başarılı olsa bile üçüncü adım onaysızsa gönderimin oluşmadığını görmelisin. Aynı tabloyu onaydan sonra alıcıyı değiştirerek yeniden dene. Eski onay işaretinin yeni hedefe taşınması beklenen bir başarı değildir. Kayıt yalnızca bir tarih ve yeşil simgeden oluşmasın. Denenen sürüm, kullanılan yapay kayıt, işlem kapsamı ve gözlenen sonuç birbirine bağlansın. Böylece daha sonra araç değiştiğinde önceki sonucun hangi koşula ait olduğunu anlayabilirsin. Gerçek kişi veya müşteri verisini test kanıtına eklemek zorunda değilsin. Bu örnek için takma bir alıcı ve gönderimsiz test yolu yeterlidir. Sonuç belirsizse bunu başarılı diye işaretlemek yerine eksik kanıt olarak bırak. Kaynak - OWASP — Authorization Cheat Sheet"
  },
  {
    "index": 6,
    "slug": "n8n-yeniden-deneme-islem-kimligi",
    "slug_tr": "n8n-yeniden-deneme-islem-kimligi",
    "title_tr": "n8n’de yeniden deneme: aynı işi ikinci kez yapmadan toparlan",
    "summary_tr": "Bir isteğin zaman aşımına uğraması, işlemin yapılmadığını kanıtlamaz. Tekrar denemeyi hata türü, bekleme süresi, işlem kimliği ve belirsiz sonuç üzerinden tasarla.",
    "word_count": 730,
    "estimated_duration_sec": 312,
    "narration_text": "n8n’de yeniden deneme: aynı işi ikinci kez yapmadan toparlan. Bir isteğin zaman aşımına uğraması, işlemin yapılmadığını kanıtlamaz. Tekrar denemeyi hata türü, bekleme süresi, işlem kimliği ve belirsiz sonuç üzerinden tasarla.. Bir otomasyon bir bildirimi göndermek ister, fakat yanıt beklerken zaman aşımına uğrar. Akışta kırmızı hata görürsün. Aynı adımı yeniden çalıştırmak kolaydır; ama bildirim ilk denemede gönderildiyse alıcı iki mesaj alabilir. Bu nedenle yeniden deneme tasarımı yalnızca kaç saniye bekleneceğiyle ilgili değildir. Aynı işin ne olduğunu, hangi durumda tekrar edilebileceğini ve belirsiz bir sonucun nasıl araştırılacağını da tanımlamak gerekir. Hata türüne göre karar ver Önce sonucu sınıflandır: geçersiz girdi, eksik yetki, hız sınırı, geçici servis sorunu veya yanıtı bilinmeyen işlem. Yanlış adres biçimini aynı veriyle beş kez göndermek genellikle çözüm üretmez. Hız sınırında ise sağlayıcının önerdiği bekleme ve kapasite koşulları önemlidir. Her hatayı tek bir genel “tekrar dene” koluna bağlamak yerine, operatörün neyi düzeltmesi gerektiğini anlayabileceği durumlar oluştur. n8n belgeleri hız sınırları için Retry On Fail ve uygun durumlarda Loop Over Items ile Wait gibi seçenekler anlatır. Bu araçların bulunması, bütün dış işlemlerin güvenle tekrarlanabileceği anlamına gelmez. Kullanacağın servisin belgelerini de incele. Özellikle veri değiştiren çağrılarda isteğin etkisini ve sağlayıcının tekrar sözleşmesini birlikte değerlendir. Sadece okuma yapan bir kaynak sorgusuyla ödeme veya mesaj gönderimi aynı riskte değildir. Deneme bütçesi belirle Bir iş için en fazla kaç deneme yapılacağını ve toplamda ne kadar süre bekleneceğini yaz. Bekleme süresini yalnızca istek başına değil, bütün işlem yaşamı boyunca düşün. Bir akışın başka bir akış tarafından yeniden başlatılması sınırsız tekrar yaratabilir. İç içe deneme kuralları varsa toplam çağrı sayısını küçük bir örnek üzerinden hesapla. İki katmanda üçer deneme, beklediğinden fazla dış istek üretebilir. Denemeler bittiğinde işi sessizce atma. İncelenecek bir kayıt oluştur: hangi hedefe gidildi, hangi hata alındı, kaç deneme yapıldı ve son bilinen durum neydi? Bu kayıtta sırlar veya gereksiz kişisel veri bulunmasın. Hata mesajını olduğu gibi günlüğe yazmadan önce içerdiği hassas alanları düşün. Bir kişisel otomasyonda bile anlaşılır bir inceleme kuyruğu, görünmez kayıptan veya sonsuz döngüden daha yararlıdır. İş kimliğini akış çalıştırmasından ayır Örnek bir haftalık özet gönderiminde iş, belirli bir haftanın belirli hedefe gönderilecek son metnidir. Aynı gönderim tekrar denendiğinde yeni iş gibi görünmemelidir. Akışın her çalıştırma kimliğini doğrudan iş kimliği olarak kullanmak bu ayrımı bozabilir. İş kaydı oluşturulurken bir kimlik üretip bütün tekrarlarla ilişkilendirmek daha anlaşılır bir modeldir. Gerçek uygulamanın eşzamanlı istekleri de nasıl ele aldığını test et. Sağlayıcı idempotency desteği sunuyorsa hangi parametreleri aynı kabul ettiğini, anahtarı ne kadar tuttuğunu ve önceki yanıtı nasıl döndürdüğünü kendi belgesinden oku. Stripe’ın idempotent istekleri bu yaklaşımın somut bir örneğini gösterir; ayrıntıları bütün API’ler için genelleme. Destek yoksa kendi gönderim kaydın da tek başına dış servisin işlemi kabul edip etmediğini her ağ kesintisinde kesin bilemeyebilir. Bu sınırı tasarımda görünür tut. Belirsiz sonuca ayrı bir kapı aç İstek gönderildikten sonra bağlantı koparsa doğrudan başarısız etiketi koymak yerine sonucun doğrulanamadığını belirtebilirsin. Sağlayıcının işlem sorgulama uç noktası, kayıt kimliği veya yönetim ekranı varsa bağımsız kontrol yap. Henüz gerçekleşmemiş gibi ikinci bir yan etki üretmeden önce bu kanıtı ara. Araştırma mümkün değilse otomatik karar yerine insan incelemesine bırakılacak dar bir yol tanımla. Bu, her akışı elle yürütmek anlamına gelmez. Son test kümesinde normal başarı, ilk çağrıda geçici hata, kalıcı hata ve yanıtı kaybolmuş kabul edilmiş işlem bulunsun. Her örnekte dış etkilerin sayısını say; yalnızca akışın tamamlandı görünmesine bakma. Buradaki deney alanı gerçek servis çağırmaz ve işlem göndermez. Kendi test ortamında hangi kayıtların oluştuğunu bağımsız okuyarak doğrula. Böylece “yeniden dene” düğmesi bir umut hareketi değil, tanımlı bir toparlanma davranışı olur. Dış etkiyi sayan bir kabul tablosu kullan Bir test gönderimini temsil eden tek bir iş kimliği seç. İlk denemede sağlayıcının isteği kabul ettiğini ama yanıtın kaybolduğunu varsay. İkinci denemede sistemin hangi kanıta bakacağını ve hangi durumda duracağını yaz. Tabloda deneme adedi ile gerçek dış etki adedi ayrı sütunlarda olsun. İki deneme yapılmış olması iki mesaj gönderilmesini normal hale getirmez. Aynı şekilde hiç yanıt alınmaması sıfır etki oluştuğunu kanıtlamaz. Sonraki satırda gerçekten yeni bir iş oluştur. Metni benzer olsa bile hedef veya zaman kapsamı değişmişse bunun yeni kimlikle nasıl temsil edildiğini açıklayabilmelisin. İş tanımı belirsizken yalnızca rastgele anahtar eklemek sorunu çözmez. Son olarak deneme bütçesi dolan kaydı inceleme kuyruğunda bul ve yeniden açma yolunu kontrol et. Kişi yeniden başlat dediğinde yeni bir iş mi oluşuyor, yoksa aynı işin durumu mu araştırılıyor? Bu farkın görünür olması, toparlanmayı hem teknik olarak hem kullanım açısından daha anlaşılır yapar. Kaynaklar - n8n: hız sınırlarını yönetme - n8n: hata iş akışları - Stripe: idempotent istekler"
  },
  {
    "index": 7,
    "slug": "react-bekleme-hata-durumlari",
    "slug_tr": "react-bekleme-hata-durumlari",
    "title_tr": "React’te bekleme ve hata durumları: kullanıcıyı yarıda bırakma",
    "summary_tr": "Bir arayüz yalnızca başarılı cevap geldiğinde tamamlanmış sayılmaz. Kullanıcının girdisini koruyan, neyin beklendiğini anlatan ve hatadan dönüş sunan küçük bir etkileşim tasarla.",
    "word_count": 710,
    "estimated_duration_sec": 304,
    "narration_text": "React’te bekleme ve hata durumları: kullanıcıyı yarıda bırakma. Bir arayüz yalnızca başarılı cevap geldiğinde tamamlanmış sayılmaz. Kullanıcının girdisini koruyan, neyin beklendiğini anlatan ve hatadan dönüş sunan küçük bir etkileşim tasarla.. Bir arama kutusu tasarladığını düşün. Kullanıcı ilk kelimeyi yazar, bir istek başlar; hemen ardından sorguyu değiştirir. İlk istek daha geç dönerse ekranda hangi sonuç görünmelidir? Bu soru, bekleme durumunun yalnızca bir dönen simgeden ibaret olmadığını gösterir. Doğru tasarım; girdiyi, istek kimliğini, görünür sonucu ve hata yolunu birlikte ele alır. Buradaki örnek bir arayüz tasarımıdır, ölçülmüş bir performans iyileştirmesi iddiası değildir. Kullanıcının neyi beklediğini tanımla “Yükleniyor” mesajı tek başına eksik olabilir. Yeni sonuç listesi mi, bir dosya mı, yoksa kaydetme işlemi mi bekleniyor? Göstergenin konumu ve metni bu soruya cevap versin. Kullanıcı arama yazarken bütün sayfayı kullanılamaz hale getirmek yerine ilgili sonuç bölgesini ele alabilirsin. Buna karşılık aynı kaydı iki kez oluşturan bir işlemde gönderim kontrolü ayrıca gerekebilir. Her bekleme durumuna aynı kilitlemeyi uygulama. Önceki sonucun ekranda kalması bazı akışlarda yararlıdır. Ancak yeni sorguya aitmiş gibi sunulursa yanıltıcı olur. Sonuçların yenilendiğini açık bir durum mesajıyla belirt ve eski verinin durumunu anlaşılır tut. Yerleşim için ayrılmış alanı korumak büyük sıçramaları azaltabilir. Görsel iskelet kullanıyorsan gerçek içerikle benzer bir alan kaplasın; sadece hareketli bir dekor olarak kalmasın. Hareket azaltma tercihi de bu tasarımın parçasıdır. Hangi mekanizmanın ne yaptığını ayır Suspense, desteklenen bir kaynak beklenirken yakınındaki fallback arayüzünü gösterebilir. Bir bileşenin içine rastgele fetch eklemek, onun otomatik olarak bu sınırı kullanacağı anlamına gelmez. Çatının veri yükleme yaklaşımını ve kullandığın bileşenin davranışını resmi belgede incele. Önce küçük bir örnekte hangi sınırın ne zaman göründüğünü gözle. Bir yükleme sorununu daha fazla sınır ekleyerek çözmeden önce gerçek veri akışını anla. Transition ise ağ hızını değiştiren bir araç değildir. Bazı güncellemelerin önceliği ve mevcut arayüzün korunmasıyla ilişkilidir. Kullanıcının metin alanındaki değeri hemen güncellenmeli; daha ağır sonuç değişimi ayrı ele alınabilir. Asenkron eylemlerden sonra hangi state güncellemesinin transition kapsamında olduğunu da kontrol et. Bir API adını kullanmış olmak, beklediğin bütün davranışların kendiliğinden oluştuğunu kanıtlamaz. Geç gelen yanıtı küçük bir testle yakala İki sorgu düşün: önce “kitap”, sonra “kitaplık”. İkinci sorgunun sonucu önce dönsün. Birinci yanıt daha sonra gelince son listeyi değiştirmemeli. İstek iptali, etkin istek kimliği veya veri katmanının sunduğu mekanizma bu soruna yardımcı olabilir; hangi yaklaşımın kullanıldığını açıkça belirle. İptal edilmiş isteğin ekranda gereksiz hata mesajı üretip üretmediğini de kontrol et. İptal her zaman kullanıcıya hata diye gösterilmemelidir. Bu testi gerçek servisi yormadan gecikmeli bir test yanıtıyla yapabilirsin. Normal sıra, ters sıra, boş sonuç ve hata için beklenen ekranı yaz. Her yanıtın yalnızca kendi sorgusuna ait sonucu değiştirdiğini doğrula. Sonra sayfadan ayrılma durumunu dene. Kullanıcı başka bölüme geçtiğinde eski isteğin görünmeyen bir bileşende beklenmeyen durum güncellemesi yaratmaması, iyi bir iptal ve yaşam döngüsü tasarımının parçasıdır. Hata sonrasında devam yolu sun Veri doğrulama hatasında kullanıcı hangi alanı değiştireceğini bilsin. Ağ sorununda aynı girdiyi koruyup yeniden deneme seçeneği sunabilirsin. Render hatası için kullanılan Error Boundary, bütün bu durumların yerine geçmez. Erişilebilir durum mesajı, odak davranışı ve hata metni birlikte kontrol edilmeli. Teknik hata yığınını ziyaretçiye göstermek yerine, neyin tamamlanmadığını ve yapabileceği güvenli sonraki adımı anlat. Son kabul turunda yalnızca fare kullanma. Klavyeyle arama yap, yeni sonuca ulaş, hatayı okuyup yeniden dene. Ekran büyütüldüğünde mesaj ve kontrol birbirinden kopuyor mu, mobilde buton görünür mü, kontrol et. Kendi inceleme listen bu incelemeyi düzenler; kodunu çalıştırmaz veya erişilebilirliği sertifikalandırmaz. Gerçek doğrulama, bu durumları çalışan arayüzde gözleyip hangi kontrolün neyi kapsadığını kaydetmekle yapılır. Bir etkileşimin durum çizelgesini hazırla Küçük bir tabloya kullanıcı eylemini, etkin isteği, görünür sonucu ve kullanılabilir kontrolü yaz. İlk satırda kullanıcı henüz yazmamış olsun. İkinci satırda arama sürsün, üçüncüde boş sonuç, dördüncüde geçerli sonuç ve beşincide ağ hatası bulunsun. Her satırda aynı düğmenin ne yaptığı belli olmalıdır. Bir durumda yeniden dene, başka durumda yeni kayıt gönder anlamına geliyorsa etiket ve davranış bu farkı açıklamalıdır. Görsel benzerlik, farklı dış etkileri gizlememelidir. Sonra sadece klavyeyle tabloda tarif ettiğin yolu izle. Sorgu yaz, sonucu bekle, boş durumda yeni sorgu gir ve hatadan sonra yeniden dene. Odak beklenmedik yere sıçrıyorsa veya hata metni kontrolün uzağında kalıyorsa bunu tasarım bulgusu olarak kaydet. Bekleme göstergesini kaldırıp aynı akışı düşünmek de yararlıdır: kullanıcı yine ne olduğunu anlayabiliyor mu? Anlamıyorsa yalnızca animasyona bağımlı bir açıklama yapmış olabilirsin. Metin, durum ve kontrol ilişkisini düzeltmek daha kalıcı bir iyileştirme sağlar. Kaynaklar - React Suspense - React useTransition - React hata sınırları"
  },
  {
    "index": 8,
    "slug": "wordpress-url-indeksleme-kontrolu",
    "slug_tr": "wordpress-url-indeksleme-kontrolu",
    "title_tr": "WordPress’te bir URL neden görünmüyor? Adım adım teşhis",
    "summary_tr": "Yeni bir SEO eklentisi kurmadan önce tek bir sayfanın durumunu incele. Yanıt kodu, erişim izni, noindex, canonical, sitemap ve Google’ın gördüğü içerik üzerinden sorunu daralt.",
    "word_count": 731,
    "estimated_duration_sec": 313,
    "narration_text": "WordPress’te bir URL neden görünmüyor? Adım adım teşhis. Yeni bir SEO eklentisi kurmadan önce tek bir sayfanın durumunu incele. Yanıt kodu, erişim izni, noindex, canonical, sitemap ve Google’ın gördüğü içerik üzerinden sorunu daralt.. Bir WordPress yazısı aramada görünmediğinde ilk tepki yeni bir SEO eklentisi kurmak olabilir. Daha yararlı başlangıç ise tek bir URL seçip ne olduğunu kanıtlarla anlamaktır. Bu yazı belirli bir sitenin incelendiğini iddia etmez; kendi sayfanda uygulayabileceğin bir teşhis sırası sunar. Düzeltmenin amacı teknik engeli bulmaktır. Kontrolleri geçmek, Google’ın sayfayı mutlaka dizine alacağı veya üst sıraya taşıyacağı anlamına gelmez. Tam olarak hangi adresi inceliyorsun? Yazının yönetim panelindeki düzenleme adresini değil, ziyaretçinin açtığı tam adresi not et. HTTP ve HTTPS, www kullanılan ve kullanılmayan biçimler ya da sonuna eklenen parametreler farklı yanıtlar üretebilir. Tahmini adresleri topluca denemek yerine menüden veya yazı listesinden gerçek bağlantıyı aç. Sonunda ulaşılan adresi ve varsa yönlendirmeyi kaydet. İlk kontrolün konusu herkesin aynı sayfadan söz ettiğinden emin olmaktır. Sayfanın giriş yapılmamış bir pencerede açıldığını da doğrula. Yönetici olarak gördüğün taslak, anonim ziyaretçi için bulunamayabilir. Bakım modu veya şifre koruması gibi özellikler de sonucu değiştirebilir. Bir ekranın görünmesiyle sunucunun doğru yanıt kodunu vermesi aynı şey değildir. Hata sayfası tasarımla güzel görünebilir; önemli olan hangi durum koduyla sunulduğu ve gerçekten istenen içeriği taşıyıp taşımadığıdır. Tarama ve dizine ekleme talimatlarını ayır WordPress’in görünürlük ayarlarını, kullanılan SEO eklentisini ve şablon çıktısını birlikte incele. Panelde bir kutunun işaretli olmaması tek başına sayfada noindex bulunmadığını kanıtlamaz. Birden fazla bileşen aynı etiketi üretiyor olabilir. Tarayıcıdaki gerçek kaynak çıktısını veya Search Console incelemesini kullanarak Google’a hangi talimatın sunulduğunu kontrol et. Aynı sorunu iki eklentiyle çözmeye çalışmadan önce mevcut çıktıyı anlamak gerekir. Robots.txt engeli ve noindex farklı araçlardır. Bir sayfanın taranmasını engellemek, içindeki dizine eklememe talimatının okunmasını da engelleyebilir. Bu nedenle gizlemek istediğin bir sayfa için rastgele iki ayarı birden açmak yerine amacını belirle. Özel içerik gerekiyorsa asıl kontrol erişim yetkisidir; arama motoru talimatını parola veya izin sistemi yerine kullanma. Kamuya açık olmaması gereken veri farklı bir sorundur. Canonical ve dil eşleştirmesini incele Sayfanın canonical adresi kendisini mi, başka bir yazıyı mı gösteriyor? Bir eski şablon veya kopyalanmış ayar bütün yazılara aynı adresi bırakmış olabilir. Ancak başka bir adres her zaman hata değildir; yinelenen sürümleri bir ana adreste toplama amacı da olabilir. Önce niyeti araştır, sonra yanlış çıktıyı üreten bileşeni düzelt. Canonical sinyaliyle sunucunun yönlendirmesini ve ziyaretçinin gördüğü bağlantıları birbiriyle uyumlu tut. Çok dilli bir sitede henüz çevrilmemiş yazı için boş İngilizce sayfa üretme. Dil bağlantısı gerçek eşdeğer içeriğe gitmeli. Ana sayfaya dönen veya Türkçe metni farklı dil etiketiyle gösteren bağlantı, kullanıcı için de yanıltıcıdır. Yazının listede görünmesi, sitemap’te yer alması ve dil eşleştirmesi aynı yayın durumunu temsil etsin. Birini düzeltirken diğerlerini eski durumda bırakmamaya dikkat et. Search Console’da iki farklı zamanı karşılaştır URL Inspection aracında kayıtlı verinin ne zamana ait olduğuna bak. Canlı test, mevcut sunucunun o anda verdiği sonucu değerlendirmeye yardımcı olur; kayıtlı sürüm ise Google’ın önceki ziyaretini anlatır. Bugün düzelttiğin bir etiketi eski sonuçta hâlâ görmek tek başına düzeltmenin başarısız olduğunu göstermez. Tarihleri not et, canlı ve kayıtlı sonucu ayrı satırlara yaz. Bu ayrım gereksiz yere aynı ayarı tekrar değiştirmeni önler. Düzeltme yaptıktan sonra yalnızca panelde kaydedildi mesajını değil, anonim sayfayı ve üretilen etiketleri yeniden kontrol et. Önbellek katmanları eski çıktıyı sunuyor olabilir. Bir değişiklik, bir bağımsız okuma ve kısa bir sonuç notu kullan. Sorun kalırsa aynı anda tema, eklenti ve alan adı ayarlarını değiştirmek yerine en küçük farklılığı araştır. Bu yöntem, hem geri dönüşü kolaylaştırır hem hangi düzeltmenin gerçekten etkili olduğunu gösterir. Teşhisi tek sayfalık bir notta bitir İncelediğin adresi en üste yaz ve altına üç zaman ekle: sorunun görüldüğü an, değişikliğin yapıldığı an ve bağımsız kontrol zamanı. Yanıt kodunu, canonical adresini ve dizine ekleme talimatını aynı notta tut. Bir eklenti ayarı değiştiyse adını ve yalnızca ilgili seçeneği belirt; bütün yapılandırmayı veya erişim bilgilerini paylaşma. Bu kayıt, birkaç gün sonra farklı bir sonuç gördüğünde neyin değişmiş olduğunu ayırmana yardım eder. Sonra sayfaya başka bir yayımlanmış sayfadan gerçekten ulaşılabildiğini kontrol et. Menüde görünmeyen ama sitemap’te bulunan bir adresle, okurun doğal biçimde keşfedebildiği içerik aynı kullanım deneyimini sunmaz. İç bağlantının metni hedefi açıklasın. Kontrol bittiğinde bütün siteyi düzelttiğini değil, bu URL için hangi engelin kaldırıldığını yaz. Başka adreslerde benzer belirti varsa aynı kök nedenin varlığını varsaymak yerine birkaç temsilî örnekte doğrula. Böylece tek sayfalık bir düzeltmeyi kontrolsüz bir site genelinde değişikliğe dönüştürmezsin. Kaynaklar - Google URL Inspection aracı - Google: tek bir sayfada arama sorunu çözme - Search Central: arama sorunlarını teşhis etme"
  },
  {
    "index": 9,
    "slug": "kaynak-degisikliklerini-izleme",
    "slug_tr": "kaynak-degisikliklerini-izleme",
    "title_tr": "Kaynak değişikliklerini izle: bağlantısı kaybolmayan bir bilgi akışı",
    "summary_tr": "Yeni bir içerik bulmak, onun doğru veya önemli olduğunu kanıtlamaz. Küçük bir kaynak listesiyle başlayan; kimlik, tarih, tekrar kontrolü ve değişiklik kaydı taşıyan bir izleme akışı tasarla.",
    "word_count": 711,
    "estimated_duration_sec": 304,
    "narration_text": "Kaynak değişikliklerini izle: bağlantısı kaybolmayan bir bilgi akışı. Yeni bir içerik bulmak, onun doğru veya önemli olduğunu kanıtlamaz. Küçük bir kaynak listesiyle başlayan; kimlik, tarih, tekrar kontrolü ve değişiklik kaydı taşıyan bir izleme akışı tasarla.. Bir bilgi akışının ilk işi iyi özet yazmak değil, bulduğu bilgiyi tekrar inceleyebileceğin biçimde saklamaktır. Özetin yanında kaynak adresi yoksa bir hafta sonra hangi cümlenin nereden geldiğini anlamak zorlaşır. Bu rehber, üç teknik kaynağı izleyen kişisel bir deney için kayıt yapısını anlatır. Dışarı otomatik yayın yapan bir sistem veya bütün kaynakların doğruluğunu garanti eden bir haber servisi kurduğunu varsaymaz. Önce bir kaynak kartı oluştur Her kaynak için adres, kısa açıklama, beklenen içerik türü ve kontrol amacı yaz. “Teknoloji haberleri” çok geniş olabilir; belirli bir ürünün sürüm notlarını izlemek daha sınanabilir bir görevdir. Kaynağın resmi olup olmadığını ve hangi sürümü anlattığını belirt. Bir topluluk yorumunu üretici belgesiyle aynı kanıt düzeyinde saklamak yerine türünü görünür tut. Bu ayrım, daha sonra özet hazırlanırken bağlamın kaybolmasını önler. Öğe düzeyinde başlık, özgün bağlantı, varsa yayıncının kimliği, yayımlanma zamanı ve senin alma zamanın ayrı alanlar olsun. Bir sayfayı bugün bulman, onun bugün yayımlandığı anlamına gelmez. Kaynak tarihi eksikse tahmin ederek doldurma; bilinmiyor olarak bırak. Zaman dilimini de not et. Farklı kaynakların saatlerini karşılaştırırken yalnızca ekrandaki tarih metnine bakmak yanlış bir sıralama yaratabilir. Okuma, karşılaştırma ve bildirim ayrı adımlardır n8n RSS Read düğümü, yayımlanan RSS adresinden veri okumak için kullanılabilir. İlk denemede yalnızca bir akışı oku ve hangi alanların geldiğini incele. Beklediğin kimlik alanı her kaynakta bulunmayabilir. Doğrudan bir özet modeline göndermek yerine örnek çıktıyı kaydet. Hangi verinin eksik olduğunu görmek, ileride modelin boşlukları akıcı ama dayanaksız cümlelerle doldurmasını önlemeye yardımcı olur. Karşılaştırma adımında “aynı öğe”, “yeni öğe”, “içeriği değişmiş öğe” ve “okunamayan kaynak” durumlarını ayır. Kaynağın erişilememesi, yeni içerik olmadığı anlamına gelmez. Aynı şekilde yeni bir kayıt bulmak da olayın yeni gerçekleştiğini kanıtlamaz. Akışın durum alanı bu ayrımları taşısın; yalnızca boş veya dolu liste döndürmesin. Kişisel bir tablo bile bu davranışı anlaşılır biçimde gösterebilir. Tekrar kontrolünü bağlama göre kur Yayıncının kalıcı öğe kimliği varsa bunu kullanma biçimini incele. Yalnızca başlığa göre karşılaştırmak, aynı başlıkla güncellenen bir belgeyi kaçırabilir. Yalnızca adrese göre karşılaştırmak ise adrese eklenen takip parametreleri yüzünden aynı sayfayı yeni sayabilir. Normalleştirme yapacaksan hangi parametreleri kaldırdığını açıklayabil; farklı içerik taşıyan adresleri yanlışlıkla birleştirme. Sonuçları birkaç gerçek örnek üzerinde gözle. HTTP kaynağı ETag sunuyorsa koşullu istek mekanizmasını değerlendirebilirsin. Sunucu uygun koşullarda 304 yanıtıyla temsilin değişmediğini bildirebilir. Bu mekanizma bir iddianın doğruluğunu veya sayfadaki hangi cümlenin önemli olduğunu söylemez. Kaynak desteğini test et ve başarısız isteği değişmemiş içerikle karıştırma. Ağ verimliliği ile editoryal değerlendirme birbirini tamamlayan ama farklı sorumluluklardır. İncelenebilir bir değişiklik notu hazırla Bir bildirimde yalnızca uzun bir AI özeti göndermek yerine önce ne değiştiğini, hangi kaynağın değiştiğini ve önceki kaydın ne zaman alındığını göster. Özet kullanıyorsan onu ayrı bir yorum katmanı olarak etiketle. Desteklenmeyen bir cümle görünür olduğunda kaynağa dönebilmeyi kolaylaştır. Karşılaştırmayı yapamadığın yerde “değişiklik doğrulanamadı” demek, kesin bir sonuç uydurmaktan daha faydalıdır. Son testte aynı kaynağı iki kez oku, örnek içeriği kontrollü biçimde değiştir ve bir kaynağı erişilemez kabul et. Her durumda hangi kaydın oluşacağını önceden yaz. Bu sayfadaki deney alanı tasarımı gözden geçirmek içindir; hiçbir kaynağa istek göndermez. Kendi izleme sisteminde kontrol sıklığını, veri saklama süresini ve bildirim alıcısını ayrıca belirlemelisin. Böylece küçük bir akış zamanla anlaşılmaz bir gürültü üreticisine dönüşmez. Örnek kayıt tablosuyla provayı tamamla Üç hayali öğe oluştur. Birincisi yeni yayımlanmış bir belge, ikincisi aynı bağlantıda güncellenmiş eski bir belge, üçüncüsü ise başlığı değişmemiş ama içeriğine not eklenmiş bir kaynak olsun. Her biri için hangi alanın karşılaştırılacağını yaz. Böylece yalnızca başlığı izleyen bir sistemin üçüncü değişikliği neden kaçırabileceğini görürsün. Ardından bir kaynağın yanıt vermediği dördüncü durum ekle. Bunun yeni içerik yok sonucu üretmemesi gerektiğini kaydına geçir. Bir hafta sonra bu tabloya döndüğünde hangi bilginin kaynaktan, hangisinin senin yorumundan geldiğini ayırabiliyor musun? Özet metni tek sütuna yığılmışsa bağlamı yeniden düzenle. Kaynak adresi, kimlik, alınma zamanı ve yorum farklı alanlarda kalsın. Kontrol aralığını da belgenin gerçek değişim hızına göre tekrar değerlendir. Bildirim sayısını azaltmak için önemli bilgiyi silmek yerine, aynı öğenin tekrarlarını tek bir incelenebilir kayıtta birleştir. Böyle bir prova, yeni araçlar eklemeden önce akışın hangi soruya cevap verdiğini netleştirir. Sonuçta topladığın şey yalnızca içerik değil, sonradan sorgulanabilir küçük bir gözlem günlüğü olur. Kaynaklar - n8n RSS Read - n8n Remove Duplicates - MDN: ETag - MDN: 304 yanıtı"
  },
  {
    "index": 10,
    "slug": "dogal-dille-kod-yazma-dogrulama",
    "slug_tr": "dogal-dille-kod-yazma-dogrulama",
    "title_tr": "Doğal dille kod yazdırmak: isteği doğrulanabilir bir işe dönüştür",
    "summary_tr": "İyi bir kod isteği yalnızca ne istediğini anlatmaz; doğru sonucu nasıl tanıyacağını da söyler. Küçük bir örnek üzerinden kabul koşulu, değişiklik incelemesi ve test akışını kur.",
    "word_count": 739,
    "estimated_duration_sec": 316,
    "narration_text": "Doğal dille kod yazdırmak: isteği doğrulanabilir bir işe dönüştür. İyi bir kod isteği yalnızca ne istediğini anlatmaz; doğru sonucu nasıl tanıyacağını da söyler. Küçük bir örnek üzerinden kabul koşulu, değişiklik incelemesi ve test akışını kur.. Doğal dille kod yazdırırken en yararlı başlangıç, “bana bir uygulama yap” demek yerine tek bir davranışı tarif etmektir. Örneğin bir okuma listesindeki boş başlıkları reddeden küçük bir form düşün. İsteğin sonunda hangi girdinin kabul edileceği ve hangi mesajın görüleceği açık olursa hem üretilen kodu incelemek hem sonucu test etmek kolaylaşır. Buradaki yaklaşım bir modelin her zaman doğru kod yazacağı iddiası değildir; insan incelemesini daha somut hale getiren bir çalışma yöntemidir. İsteği bir örnekle daralt “Formu iyileştir” yerine şu davranışı tarif et: Başlık boşsa kayıt oluşturulmasın, alanın yanında anlaşılır bir mesaj gösterilsin ve kullanıcının yazdığı diğer bilgiler korunsun. Sonra iki örnek ekle. “Akşam okuması” geçerli olsun; yalnızca boşluk içeren başlık geçersiz sayılsın. Bu iki örnek, görünüşte küçük bir ayrıntının veri davranışını nasıl değiştirdiğini gösterir. Kenar durumunu belirtmediğinde araç kendi varsayımını yapabilir. Mevcut projede hangi dilin, çatının ve form yaklaşımının kullanıldığını da incele. Yeni bir özellik, var olan yapının tamamını yeniden yazma gerekçesi değildir. Korunacak dosyaları, mevcut kullanıcı değişikliklerini ve gerçek servislere dokunulup dokunulmayacağını açıkça belirt. Bu sınırlar yalnızca güvenlik için değil, incelemenin anlaşılır kalması için de önemlidir. Tek bir değişikliğin sonucunu görmek, aynı anda beş mimari kararın etkisini ayırmaktan daha kolaydır. Kabul koşulunu koddan önce yaz Bir kontrol listesi üç soruya cevap versin: Normal girdide ne olur, yanlış girdide ne olur ve işlem yarıda kalırsa ne korunur? Örnek formda geçerli başlık kaydedilir, boş başlık reddedilir ve başarısız ağ isteğinde metin kaybolmaz. Bunları kullanıcı davranışı olarak yaz; uygulanacak fonksiyonun adını testin tek amacı haline getirme. Aynı davranışı farklı ama doğru bir uygulama da sağlayabilsin. Henüz sunucu bağlantısı yoksa bunu ayrıca belirt. Tarayıcıda bir başarı mesajı göstermek, gerçek kaydın veritabanına yazıldığını kanıtlamaz. Prototipte hangi parçanın örnek olduğunu görünür bırak. Daha sonra gerçek servis bağlandığında ayrı bir uçtan uca kontrol yap: kaydı oluştur, bağımsız okumayla varlığını doğrula ve test için oluşturduğun kaydı tanımlı geri alma yöntemiyle temizle. Üretim verisini öğrenme deneyi olarak kullanma. Değişiklik farkını oku Üretilen yanıtın açıklamasına güvenmekle yetinme; değişen dosyaları incele. İstenen form düzenlemesi yanında paket listesi, yetki kuralı veya yayın ayarı değişmiş mi? Yeni bir ağ isteği eklenmişse hangi adrese hangi veriyi gönderdiğini araştır. Anlamadığın bir değişikliği yalnızca testler yeşil göründüğü için onaylama. Önce amacını ve beklenen etkisini açıklat, ardından kodun bu açıklamayla uyuşup uyuşmadığını kontrol et. İnceleme çok büyüdüyse işi böl. Önce veri doğrulama, sonra hata görünümü, ardından erişilebilirlik gibi küçük adımlar seçebilirsin. Ancak bölmek, parçalar arasındaki ilişkiyi unutmak anlamına gelmesin. Form alanı hatayı gösterirken sunucu da aynı geçersiz girdiyi reddediyor mu, birlikte düşün. Kullanıcının tarayıcı kontrollerini atlayabileceğini kabul eden bir sınır tasarla. Görsel bir uyarı veri bütünlüğünün yerine geçmez. Çalıştırılan kontrolü dürüstçe raporla Son notta “tamamlandı” yerine doğrulanan davranışı anlat: Boş başlık reddedildi, geçerli örnek kaydedildi, ağ hatasında giriş korundu. Mobil tarayıcıyı veya gerçek servis bağlantısını test etmediysen bunu ayrı yaz. Böyle bir rapor, bir sonraki çalışmanın nereden başlayacağını gösterir. “Her şey çalışıyor” gibi geniş bir ifade ise kontrol edilmemiş alanları görünmez hale getirir. Deney alanını ve kendi çalışma notlarını kendi küçük görevin için doldurabilirsin. Liste kod çalıştırmaz; örneklerini ve inceleme adımlarını düzenlemene yardım eder. Son olarak sorunu yeniden üretmek için gereken en küçük notu sakla: ortam, girdi, beklenen sonuç, gözlenen sonuç ve yapılan düzeltme. Bu not, yalnızca bugünkü hatayı çözmez; aynı davranışın sonraki sürümde bozulduğunu anlaman için de somut bir referans oluşturur. İsteğin için küçük bir inceleme formu Bir sonraki kod görevinden önce dört satır doldur: değişecek davranış, korunacak davranış, örnek girdi ve başarı kanıtı. Örneğin başlık doğrulaması eklenirken mevcut kayıtların sıralaması korunacaksa bunu açıkça yaz. Böylece aracın formu iyileştirirken liste davranışını da değiştirmesinin görev dışı olduğunu anlayabilirsin. Korunacak özelliklerin belirtilmesi, işi gereksiz büyütmeden sınırı görünür kılar. Her ayrıntıyı önceden tahmin etmek zorunda değilsin; sonucu değiştiren belirsizliği seçmen yeterlidir. Değişiklik sonrasında aynı formun altına gözlenen sonucu ekle. Bir test çalıştırılamadıysa nedeni ve hangi davranışın belirsiz kaldığı ayrı dursun. “Muhtemelen doğrudur” ifadesini geçen kontrol gibi işaretleme. Başka bir kişi çalışmayı devraldığında bu iki satır, uzun bir sohbet geçmişinden daha hızlı yön verebilir. Geri alma yolunu da ekle: hangi dosya veya kayıt değişti, önceki sürüm nerede duruyor ve geri dönüşün beklenen etkisi ne? Bu alışkanlık, doğal dille verilen isteği somut ve incelenebilir bir geliştirme işine dönüştürür. Kaynak - GitHub: kod önerilerini sorumlu kullanma ve sınırlamalar Bu yazıdaki form örneği açıklayıcı bir tasarımdır; gerçek bir kullanıcı projesinde ölçülmüş başarı veya hız artışı olarak sunulmaz."
  },
  {
    "index": 11,
    "slug": "react-yapay-zeka-bilesen-sozlesmesi",
    "slug_tr": "react-yapay-zeka-bilesen-sozlesmesi",
    "title_tr": "AI çıktısını arayüze dönüştürmek: serbest HTML yerine bileşenler",
    "summary_tr": "Modelin ürettiği veriyi doğrula, izin verilen bileşenlere eşleştir ve yükleme ile hata durumlarını tasarla.",
    "word_count": 750,
    "estimated_duration_sec": 321,
    "narration_text": "AI çıktısını arayüze dönüştürmek: serbest HTML yerine bileşenler. Modelin ürettiği veriyi doğrula, izin verilen bileşenlere eşleştir ve yükleme ile hata durumlarını tasarla.. Bir modelin cevabını ekranda göstermek ile modelin arayüz kodunu çalıştırmasına izin vermek farklı kararlardır. Kişisel bir uygulamada ilk deneme için küçük, önceden tanımlanmış bileşenler ve doğrulanmış veri daha denetlenebilir bir başlangıç sunar. Bu yazı üretimde ölçülmüş bir performans sonucu değil, bir uygulama tasarımıdır. Önce veri sözleşmesi Örneğin model yalnızca “özet”, “kontrol listesi” veya “kaynak kartı” türlerinden birini önerebilsin. Her türün zorunlu alanlarını, metin uzunluklarını ve izin verilen bağlantı biçimlerini belirle. Bilinmeyen türü ya da geçersiz adresi render etmek yerine reddet. Modelin verdiği bileşen veya paket adını dinamik olarak çalıştırma. React metin içeriğini normal şekilde render etmeyi destekler. Ham HTML ekleme özelliği ise güvenilmeyen veri için ayrıca risk taşır. Veriyi doğrulamak tek başına her HTML kullanımını güvenli yapmaz; ham HTML gerekmiyorsa bu yolu hiç açmamak daha anlaşılırdır. Bekleme ve hata da tasarımın parçası Yüklenirken hangi bölümün hazır olmadığını göster. Suspense, destekleyen veri kaynağı veya tembel yüklenen bileşen için fallback sunabilir; sıradan bir effect içindeki isteği kendiliğinden yönetmez. Transition kullanmak da bütün asenkron işi otomatik olarak güvenli veya hızlı yapmaz. Kontrollü metin girişinin güncellemesini transition içine taşımak yerine yazma hissini koru. Kabul testini küçük tut Geçerli bir kart, eksik zorunlu alan, aşırı uzun metin ve izin verilmeyen bağlantıyla dene. Ekran okuyucu için bekleme ve hata mesajlarını kontrol et. Bir kullanıcı işlemi veri değiştiriyorsa onayı ayrıca al; model çıktısını yetki olarak kullanma. Aşağıdaki kontrol listesi arayüz sözleşmesini gözden geçirmek içindir, çalışan kod üretmez. Üç kartlık bir sözlükle başla İlk prototipte bir özet kartı, kontrol listesi ve kaynak kartı yeterli olabilir. Özet kartının başlığı ve kısa metni, kontrol listesinin sınırlı sayıda maddesi, kaynak kartının ise açıklaması ve adresi olsun. Modelin bu sözlüğü genişletmesine izin verme. İstenen tür listede yoksa uygulama bunu desteklenmeyen çıktı olarak ele alsın. Böylece hangi ekranların oluşabileceği geliştiricinin ve kullanıcının anlayabileceği bir sınır içinde kalır. Örneğin kaynak kartındaki adresin gerçekten izin verdiğin bağlantı biçiminde olduğunu kontrol et. Kartın üzerinde yazan metin ile gittiği hedef farklıysa bunun okur için ne anlama geldiğini düşün. Bir şema yalnızca dize uzunluğunu denetliyorsa bağlantının güvenilir olduğunu kanıtlamaz. Uygulamanın amacı gerektiriyorsa bilinen alan adları veya açık bir dış bağlantı göstergesi gibi ek kurallar tasarla. Görünüm ile yetkiyi aynı veri alanına koyma Bir kartın üzerinde “Gönder” yazması onun mesaj göndermeye yetkili olduğu anlamına gelmesin. Görsel öneri, uygulanacak işlem ve kullanıcının kararı ayrı katmanlar olarak kalmalı. Model yalnızca bir eylem taslağı hazırlıyorsa, arayüz bunu son hedef ve içerikle birlikte göstermeli. Kullanıcı etkileşimi olmadan veri değiştiren bir çağrıya dönüşmemesi, özellikle sonraki sürümlerde yeni araçlar eklenirken önemlidir. Bu ayrım küçük bir alışveriş listesinde bile işe yarar. Model ürün önerisi hazırlayabilir; ama öneriyi mağazaya sipariş olarak iletmek başka bir işlemdir. Aynı kart sözlüğünün iki davranışı sessizce birleştirmesine izin verirsen arayüz, kullanıcının beklemediği bir etki üretir. Önceki çalışmanın yetkileri yeni bileşenlerin ihtiyacını otomatik karşılamaz; sınırı her yeni işlemde kontrol et. Bozuk çıktıyı tasarımın gerçek bir durumu yap Yalnızca iyi görünen örneğe göre ekran hazırlama. Başlıksız kart, boş madde listesi, çok uzun kelime, yanlış URL ve bilinmeyen tür için beklediğin davranışı yaz. Hata kartının boyutu bütün sayfayı sıçratmamalı; kullanıcının yazdığı soru kaybolmamalı. Klavyeyle yeni sonuca ulaşılabiliyor mu ve durum mesajı ekran okuyucuya anlaşılır biçimde iletiliyor mu, ayrıca dene. Son aşamada bir sürüm notu tut: desteklenen türler, sınırlar ve reddedilen örnekler. Model veya istem değiştiğinde aynı örnekleri yeniden çalıştır. Amaç her çıktıyı kabul etmek değil, uygulamanın neyi kabul ettiğini tutarlı biçimde açıklayabilmektir. Bu sözleşme, etkileyici bir ilk demodan bakım yapılabilen küçük bir ürüne geçerken en kullanışlı tasarım belgelerinden biri olur. Bileşen sözleşmesini bir tabloya dök Üç satırlı bir tablo hazırla: özet, kontrol listesi ve kaynak kartı. Her satıra zorunlu alanları, en büyük metin boyutunu ve bozuk veride gösterilecek sonucu yaz. Sonra bu tablodan haberi olmayan birinin aynı çıktıyı üretmeye çalıştığını düşün. Tür adları veya alan anlamları belirsizse bunu modelin tahmin etmesini beklemek yerine sözleşmede düzelt. Açık bir alan adı, uzun bir isteme eklenen tekrar cümlelerinden daha faydalı olabilir. Deneme verisindeki başlığa bir HTML etiketi koy ve ekranda metin olarak kalıp kalmadığını kontrol et. Kaynak adresine izin vermediğin bir biçim ekle; bağlantının hiç oluşturulmamasını beklediğini not et. Bu örnekleri test ortamında kullan ve gerçek kullanıcı verisiyle birleştirme. Son olarak kartın anlaşılır adını, klavye sırasını ve boş sonuç metnini gözden geçir. Sözleşme yalnızca veri doğrulamayı değil, geçerli bilginin insanlar tarafından kullanılabilmesini de kapsamalıdır. Kaynaklar - React: ham HTML ekleme - React: Suspense - React: useTransition Sıradaki adım n8n’i yerelde öğrenmek: ücretsiz lisans, gerçek maliyetler rehberiyle devam edebilir veya bu yazının çalışma notlarını PDF olarak indirebilirsin."
  },
  {
    "index": 12,
    "slug": "yapay-zeka-aramalarinda-gorunurluk",
    "slug_tr": "yapay-zeka-aramalarinda-gorunurluk",
    "title_tr": "AI aramalarında görünürlük: sağlam SEO ile başla",
    "summary_tr": "Özel bir “AI hilesi” yerine erişilebilir, kaynaklı ve cevap veren içerik üret. Sonucu bir görünürlük garantisiyle karıştırma.",
    "word_count": 739,
    "estimated_duration_sec": 316,
    "narration_text": "AI aramalarında görünürlük: sağlam SEO ile başla. Özel bir “AI hilesi” yerine erişilebilir, kaynaklı ve cevap veren içerik üret. Sonucu bir görünürlük garantisiyle karıştırma.. Bir yazının AI yanıtlarında anılması garanti edilemez. Google’ın AI özellikleri belgesi, bu alanlarda görünmek için normal arama gerekliliklerinin üzerine ek bir teknik zorunluluk koymadığını açıklar. Dolayısıyla işe “gizli AI etiketi” arayarak değil, okurun sorusunu ve sayfanın erişilebilirliğini düzelterek başla. Önce cevap, sonra ayrıntı Başlık belirli bir sorunu söylesin. İlk paragraf bu soruya kısa bir yanıt versin; sonraki bölümler sınırları, örneği ve kaynakları açsın. Bu düzen editoryal bir tercihtir, sıralama faktörü veya alıntılanma sözü değildir. Aynı konuyu küçük kelime değişiklikleriyle çoğaltmak yerine tek güçlü rehberde birleştir. Teknik temeli kontrol et Sayfa gerçekten yayımlanmış mı, doğru durum kodunu veriyor mu ve gezinmeden ulaşılabiliyor mu? Canonical, sitemap ve dil bağlantıları aynı gerçek sayfayı tarif etmeli. İngilizce karşılığı yoksa boş veya Türkçe kopya bir İngilizce URL üretme. Yapısal veri, sayfada görünmeyen ödül, deneyim veya puan iddiaları eklemek için kullanılmamalı. Kaynağın güvenini görünür yap Bir teknik iddiayı mümkünse üreticinin belgesine bağla. İnceleme tarihini gerçek değişiklikle güncelle; yalnızca “yeni” görünmek için tarih oynama. Deneyim diye sunduğun şeyin gerçekten yaptığın bir çalışma olduğundan emin ol. Örnek senaryoyu müşteri sonucu gibi anlatma. Neyi ölçebilirsin? Search Console üzerinden dizine eklenme ve sorgu görünürlüğünü izle. Bir testte sayfanın alıntılanması sürekli görünürlük kanıtı değildir. Aşağıdaki kontrol listesi yayıma hazırlığı değerlendirir; arama sonucu veya AI cevabı vaat etmez. Bir yazı için küçük bir kanıt haritası çıkar İlk olarak yazının cevaplamaya çalıştığı soruyu tek cümleye indir. Örneğin “AI aramalarında görünmek için yeni bir dosya zorunlu mu?” sorusu, “Yapay zekânın geleceği” başlığından daha sınanabilir bir niyet taşır. Cevabın yanına resmi kaynağı koy ve hangi bölümünün bu cevabı desteklediğini not et. Okurun ihtiyacı kısa cevap ise en başa sonuç cümlesini, devamına sınırları yerleştir. Sonra taslakta geçen her iddiayı aynı türden sanma. Üreticinin belgesindeki teknik koşul, kendi deneyiminden gelen gözlem ve ileride denemek istediğin öneri ayrı şeylerdir. Bunları dilde ayırabilirsin: “Belgede belirtiliyor”, “Bu örnekte gözledim” veya “Şu denemeyle başlayabilirsin” gibi ifadeler, okurun kanıt düzeyini anlamasını sağlar. Yapmadığın deneyi kişisel başarı hikâyesi gibi yazma. Aynı soruyu beş farklı yazıya bölmek yerine derinleştir Arşivinde çok benzeyen başlıklar varsa her birini yeni tarih ve görselle yayımlamak cazip gelebilir. Önce okurun bu sayfalar arasında anlamlı bir fark görüp görmediğine bak. Aynı cevabı tekrar eden taslaklar kaynakları ve bakım zamanını dağıtır. Bir ana rehber seçip gerçekten farklı alt soruları ayrı içeriklere dönüştürmek daha anlaşılır bir okuma yolu kurar. Örneğin görünürlük rehberi editoryal ilkeleri anlatırken, WordPress kontrol listesi tek bir URL’nin erişim sorununu çözebilir. İlki neden ve yaklaşımı, ikincisi somut teşhisi taşır. Bu iki içerik birbirinin tekrarına dönüşmeden bağlantılanabilir. İç bağlantının metni de bunu açıklasın; her yerde aynı belirsiz “buraya tıkla” çağrısını kullanma. Okur sonraki sayfada ne bulacağını bilsin. Yapısal veriyi kısa bir kayıt formu gibi incele Başlık, yazar, görsel ve tarihin gerçekten bu sayfaya ait olduğunu kontrol et. Başka bir yazıdan kalan açıklama veya aynı temsili görsel, teknik olarak geçerli JSON içinde de yanlış bağlam yaratabilir. Bir alanı yalnızca araç daha çok puan göstersin diye doldurma. Gerekli olmayan bir iddiayı çıkarmak, doğru görünmesi için yeni bir iddia uydurmaktan daha iyi bir editoryal karardır. Son değerlendirmeyi iki ayrı notta tut: “Sayfa doğru anlatılıyor mu?” ve “Okur bu içerikle bir işini yapabiliyor mu?” İlkini şema ve teknik kontroller, ikincisini örnekler, anlaşılır anlatım ve gerçek geri bildirim destekler. Sayfa hızının iyi olması yanlış bir açıklamayı düzeltmez; iyi bir yazı da erişilemeyen URL’yi kendiliğinden görünür yapmaz. İki tarafı birlikte sürdürmek gerekir. Bir paragrafı kaynakla birlikte yeniden oku Taslağından tek bir teknik paragraf seç ve yanındaki bağlantıyı aç. Kaynak aynı ürün sürümünü mü anlatıyor, söylediğin cümleyi gerçekten destekliyor mu ve önerinin geçerli olduğu koşulları belirtmiş misin? Yanıt belirsizse paragrafı daha iddialı yapmak yerine kapsamını daralt. Bir önerinin sınırlarını yazmak metni zayıflatmaz; okurun hangi durumda kullanabileceğini anlamasını sağlar. Kaynağın dili farklıysa çeviride koşul bildiren sözcükleri kaybetmediğini kontrol et. Sonra bu paragrafı teknik bilgisi daha az bir okura anlatır gibi sadeleştir. İlk cümlenin soruya cevap verip vermediğine, örneğin yeni bir bilgi ekleyip eklemediğine ve son bağlantının anlamlı bir sonraki adım olup olmadığına bak. Bir etiket veya şema alanı eklemek bu editoryal işi senin yerine yapmaz. Yazıyı tamamladığında hem kaynak haritasını hem gerçek inceleme tarihini sakla; sonraki güncelleme sırasında neyi yeniden kontrol edeceğin böylece belli olur. Kaynaklar - Google: AI özellikleri ve web sitesi - Search Essentials - Faydalı, güvenilir, insan odaklı içerik Sıradaki adım AI çıktısını arayüze dönüştürmek: serbest HTML yerine bileşenler rehberiyle devam edebilir veya bu yazının çalışma notlarını PDF olarak indirebilirsin."
  },
  {
    "index": 13,
    "slug": "yapay-zeka-token-maliyet-hesabi",
    "slug_tr": "yapay-zeka-token-maliyet-hesabi",
    "title_tr": "AI çağrısının maliyetini tahmin et: tokenlar ve sınırlar",
    "summary_tr": "Model fiyatını ezberlemek yerine kendi kullanım senaryonu hesapla; basit hesabın dışında kalan giderleri gör.",
    "word_count": 745,
    "estimated_duration_sec": 319,
    "narration_text": "AI çağrısının maliyetini tahmin et: tokenlar ve sınırlar. Model fiyatını ezberlemek yerine kendi kullanım senaryonu hesapla; basit hesabın dışında kalan giderleri gör.. Bir otomasyona model bağlamadan önce “ayda kaç çağrı, çağrı başına ne kadar veri?” sorularını cevapla. Buradaki hesaplayıcı yalnızca giriş ve çıkış tokenlarının sade bir tahminidir. Başlangıçtaki birim fiyatlar varsayımsaldır; herhangi bir modelin güncel tarifesi değildir. Hesabın mantığı Çağrı sayısını, çağrı başına ortalama giriş tokenı ve giriş birim fiyatıyla çarp. Aynı hesabı çıkış için yapıp iki tutarı topla. Birim fiyat bir milyon token için verilmişse her iki çarpımı da bir milyona böl. Aşağıdaki örnek değerler 1.000 çağrı için 2,70 USD üretir; bu bir sağlayıcı teklifi değildir. Token sayısı kelime sayısına eşit değildir. Dil, kod ve medya türü sonucu değiştirebilir. Gerçek isteklerin kullanım alanlarını sağlayıcının yanıtından kontrol et. Gemini belgelerindeki token açıklamalarını referans alırken kullandığın API ve sürümün alan adlarını ayrıca doğrula. Basit hesap neyi kapsamaz? Önbellek, düşünme tokenları, araç kullanımı, medya işleme, yeniden denemeler, barındırma ve vergiler ayrı kalabilir. Kullandığın model bunları farklı fiyatlandırabilir. Bu nedenle bu araç bir fatura simülatörü değildir. Bütçeyi uygulamada koru Önce küçük bir örnek kümesiyle ortalama tüketimi ölç. Ardından istek başına çıktı sınırı, günlük çağrı limiti ve zaman aşımı belirle. Kullanıcı hatasında tekrar denemekle geçici servis hatasında tekrar denemeyi ayır. Sınır dolduğunda sessizce başka ücretli modele geçme; açık bir durum göster. Model ve fiyat değişikliklerinde tahminini güncelle. Hesaplayıcıdaki değerler bu açık sayfada kalır. Etkileşim sayacı bu sayıları veya yazdığın veriyi kaydetmez. Sayıları küçük bir deneyden çıkar Bir kişisel not asistanı tasarladığını düşün. Üç kısa not, üç orta uzunlukta belge ve beklenenden uzun bir örnek seç. Gerçek özel kayıtlar yerine aynı yapıyı taşıyan deneme metinleri kullan. Her istek için giriş büyüklüğünü, istenen yanıt uzunluğunu ve dönen kullanım bilgisini not et. Kısa bir sorunun uzun bir sistem talimatıyla gönderildiğinde küçük kalmayabileceğini bu tabloda görebilirsin. Sonra aynı soruyu farklı yanıt sınırlarıyla karşılaştır. Daha kısa bir yanıtın yeterli olduğu bir işte uzun çıktı istemek, okurun işini kolaylaştırmadan tüketimi artırabilir. Buna karşılık çok kısa sınır yüzünden yarım kalan bir sonuç yeniden istek gerektiriyorsa toplam gider düşmeyebilir. Amaç yalnızca token sayısını azaltmak değil, işe yarayan cevabın toplam maliyetini anlamaktır. Bu deney bir modelin kalitesini bütün görevlerde ölçtüğünü iddia etmez. Birim hataları tahminden daha çok zarar verebilir Tarifede bin token mı, bir milyon token mı yazdığını kontrol et. Dolar ve başka bir para birimini aynı sütunda toplama. Bir iş akışında iki model varsa her modelin giriş ve çıkış fiyatını ayrı satırda tut. Ondalık ayırıcıyı ve fiyatın hangi tarih için geçerli olduğunu not et. Birim dönüşümünü formülün içinde gizlemek yerine hesapta görünür bırakmak, küçük bir tabloyu daha denetlenebilir yapar. Örnek bir aylık hesapta önce planlanan istek sayısını yaz. Ardından çağrı başına giriş ve çıkış tahminlerini ekle. Son olarak tekrarlar ve altyapı için ayrı bir satır aç. Bunları tek bir token katsayısına sıkıştırmak yerine ayrı göstermek, sonradan hangi varsayımın yanlış çıktığını anlamayı sağlar. Vergi ve kur etkisini dahil etmiyorsan toplamın yanında bu sınırı açıkça belirt. Tahmin ile sınır aynı araç değildir Bir elektronik tabloda bütçenin görünmesi, uygulamanın o bütçeyi koruduğu anlamına gelmez. İşlem başlamadan önce kalan hakkı kontrol eden bir kural ve tüketim kaydını güncelleyen bir akış gerekir. Birden fazla istek aynı anda geliyorsa hepsinin aynı son hakkı kullanmadığından emin ol. Kişisel bir uygulamada bile hata mesajını, bekleme durumunu ve yeniden deneme davranışını düşün. İlk haftanın ardından beklenen ve gözlenen toplamı karşılaştır. Farkı yalnızca modele bağlama: uzun bağlam, yinelenen talimat, başarısız deneme veya beklenmedik kullanıcı davranışı etkili olabilir. Bir değişiklik yaptığında yeni örnek kümesiyle yeniden ölç. Böylece hesaplayıcı bir kere doldurulup unutulan sayı yerine tasarımdaki tercihleri görünür kılan bir çalışma notuna dönüşür. Küçük bir duyarlılık deneyi Hesaplayıcıda önce yalnızca çağrı sayısını iki katına çıkar. Diğer değerler sabitken toplamın nasıl değiştiğini gözle. Sonra başlangıç değerlerine dön ve sadece çıktı uzunluğunu artır. Bu deney, hangi varsayımın sonuç üzerinde daha etkili olduğunu anlamana yardım eder. İki değişkeni aynı anda değiştirdiğinde gözlediğin farkın nedenini ayırmak zorlaşır. Her denemenin varsayımını bir satırla yazmak yeterlidir; karmaşık bir rapor hazırlamana gerek yok. Son olarak bir isteğin yeniden denendiği durumu düşün. Başarısız görünen ilk çağrının ücretlendirilip ücretlendirilmediği sağlayıcının koşullarına bağlı olabilir. Bu yüzden tekrar sayısını yalnızca başarılı yanıt adedi üzerinden hesaplama. Gerçek kullanım kaydını inceleyebildiğin küçük bir test seç ve sağlayıcının raporuyla kendi sayacını karşılaştır. Fark varsa otomatik olarak fiyat hatası varsayma; hangi isteklerin kapsama girdiğini araştır. Bu çalışma, belirsiz bir güven payını somut bir kullanım varsayımına dönüştürür. Kaynak - Google AI: token kavramı ve kullanım Sıradaki adım AI aramalarında görünürlük: sağlam SEO ile başla rehberiyle devam edebilir veya bu yazının çalışma notlarını PDF olarak indirebilirsin."
  },
  {
    "index": 14,
    "slug": "yapay-zeka-akisinda-insan-onayi",
    "slug_tr": "yapay-zeka-akisinda-insan-onayi",
    "title_tr": "Bilgi toplayan bir AI akışı: insan onayı nerede olmalı?",
    "summary_tr": "Kaynağı saklayan, belirsizliği gösteren ve işlem yapmadan önce duran küçük bir akış tasarla.",
    "word_count": 757,
    "estimated_duration_sec": 324,
    "narration_text": "Bilgi toplayan bir AI akışı: insan onayı nerede olmalı?. Kaynağı saklayan, belirsizliği gösteren ve işlem yapmadan önce duran küçük bir akış tasarla.. Bir bilgi asistanını “her şeyi kendi yapan ajan” diye tasarlamak yerine dar bir soruyla başlat: Seçtiğim kaynaklardan bir özet çıkarabilir mi ve hangi cümleyi nereden aldığını gösterebilir mi? Bu yazı uygulanmış bir müşteri projesi veya ölçülmüş başarı oranı değil, kişisel deney için bir tasarım örneğidir. Toplama ile işlem yapmayı ayır Örneğin üç teknik belgenin bağlantısını elle seç. Akış belge başlığını, adresini ve kontrol tarihini saklasın. Bir sayfanın içinde yazan talimat, sisteminin yetkisi olmasın. Modelin ürettiği özet de doğrulanmış kaynak yerine geçmesin. Sonraki adımda iddiaları kaynaklarıyla yan yana göster. Kaynağı bulunmayan cümleyi sessizce tamamlamak yerine “doğrulanmadı” diye işaretle. Birbiriyle çelişen belgelerde sürüm ve tarih farkını araştır. Özet akıcı olduğu için doğru olduğunu varsayma. Onay kapısını doğru yere koy n8n’in insan denetimli araç çağrısı yaklaşımı, araç çalışmadan önce kişinin onay veya ret verebilmesine dayanır. Bildirim göndermek, kayıt değiştirmek ya da dışarı içerik yayımlamak gibi etkisi olan adımları bu kapının arkasına koy. Onay ekranında hedefi, önerilen işlemi ve kullanılacak veriyi göster. Yalnızca akışın başında “devam” demek, sonradan değişen bir işlem için anlamlı onay değildir. Ret yanıtı ve süre aşımı da ayrı sonuçlar olsun. Yeniden denemelerin aynı e-postayı iki kez göndermemesi için işlem kimliği ve tekrar kontrolü tasarla. Küçük bir kabul testi Doğru bir belge, alakasız bir belge ve yanıltıcı talimat içeren bir örnekle dene. Beklenen sonuçları önceden yaz. Bu test başarı yüzdesi üretmek için değil, yanlış kaynağı kabul etme ve izinsiz işlem yapma gibi hataları görünür kılmak içindir. Alttaki adımlar yalnızca açıklayıcıdır; canlı bir otomasyon başlatmaz. Onay ekranını bir karar belgesi gibi düşün Örnek senaryoda akış, üç ürün belgesinden haftalık bir not hazırlasın. İlk ekranda yalnızca “Onaylıyor musun?” yazması yeterli değildir. Hangi kanala gönderileceğini, son metni, dayanak kaynaklarını ve ek dosya olup olmadığını birlikte göster. Kişi yanlış bir hedef görüyorsa metni beğenmesi işlem için yeterli gerekçe değildir. Metin düzenlendiğinde kararın hangi sürüm için geçerli olduğunu açık tut. Burada yararlı ayrım, hazırlık kaydı ile uygulama kaydıdır. Hazırlık kaydı kaynakları ve öneriyi taşır. Uygulama kaydı ise hangi önerinin, hangi hedef için ve hangi karardan sonra işlendiğini anlatır. Bir kişisel deneyde bu ayrımı küçük bir tabloyla kurabilirsin. Büyük bir kontrol paneli tasarlamadan önce aynı kaydın iki kez gönderilmesini nasıl fark edeceğini açıklayabilmen daha önemlidir. Sessizlik, izin verilmiş bir işlem değildir Bildirim ulaştıktan sonra kişi yanıt vermeyebilir. Bağlantı süresi dolabilir veya onay ekranındaki bilgiler artık güncel olmayabilir. Bu durumları otomatik kabul saymak yerine bekleyen, reddedilen ve süresi dolan kayıtlar olarak ayır. Süresi dolan bir kararı yeniden açarken son metni ve hedefi tekrar göster. İşlemin acil olması, görünmeyen değişikliklerin önceki onaya eklenmesini anlaşılır kılmaz. Bir başka zor durum, dış servisin isteği aldıktan sonra yanıtın kaybolmasıdır. Bu durumda akışın “başarısız” görünmesi, mesajın kesinlikle gönderilmediği anlamına gelmez. Tekrar denemeden önce sağlayıcının işlem sorgulama veya tekrar engelleme desteğini incele. Testte aynı hazırlık kaydını iki kez kullan; beklediğin sonuç iki mesaj değil, aynı işin önceki durumunun anlaşılması olsun. İyi bir test kümesi üç tür yanlışlığı ayırır Önce içerik yanlışlığını dene: kaynakta bulunmayan bir tarih veya özellik eklenirse özet bunu işaretleyebiliyor mu? Sonra yetki yanlışlığını dene: belge içindeki bir komut, akışa yeni bir hedef ekletebiliyor mu? Son olarak durum yanlışlığını dene: ret verilmiş bir kayıt yeniden başlatıldığında gönderim yoluna düşüyor mu? Bu ayrımlar, sorunu yalnızca modelin dil yeteneğine bağlamadan incelemeyi sağlar. Her örneğin yanına beklenen sonucu ve gözlenen davranışı yaz. Başarılı çalışan örneği de sakla; yalnızca hataları toplamak, düzeltmenin normal akışı bozduğunu kaçırmana yol açabilir. Bir kontrolün neden var olduğunu kısa bir cümleyle açıklayamıyorsan onu yeniden tasarla. Amaç otomasyonu yavaşlatmak değil, insanın gerçekten anlamlı bir karar verebildiği noktayı görünür kılmaktır. Onay kaydını bir örnekle sınama Kâğıt üzerinde üç hazırlık kaydı oluştur. Birincisi doğru hedefe giden kısa bir not, ikincisi yanlış hedefe giden aynı metin, üçüncüsü doğru hedefe giden ama kaynağı eksik bir öneri olsun. Her biri için kişinin ne görmesi ve hangi kararı verebilmesi gerektiğini yaz. Böylece onay ekranını yalnızca metin düzenleme kutusu olarak tasarlayıp tasarlamadığını fark edersin. Hedef ve kanıt görünmüyorsa kararın önemli bir bölümü saklı kalmış demektir. Ardından ilk kaydın onayından sonra metne yeni bir cümle eklendiğini varsay. Eski kararın yeni metne uygulanmamasını sağlayacak sürüm ilişkisini çiz. Gerçek bir veritabanı kurman şart değil; kayıt kimliği, sürüm ve karar sütunlarından oluşan örnek tablo yeterli olabilir. İptal edilen veya süresi dolan kayıtları da tabloda tut. Sonuçları bu şekilde görünür hale getirmek, ileride otomasyon düğümlerini kurarken hangi dalların eksik olduğunu anlamayı kolaylaştırır. Kaynak - n8n: araçlarda insan onayı Sıradaki adım AI çağrısının maliyetini tahmin et: tokenlar ve sınırlar rehberiyle devam edebilir veya bu yazının çalışma notlarını PDF olarak indirebilirsin."
  },
  {
    "index": 15,
    "slug": "n8n-yerel-kurulum-ilk-akis",
    "slug_tr": "n8n-yerel-kurulum-ilk-akis",
    "title_tr": "n8n’i yerelde öğrenmek: ücretsiz lisans, gerçek maliyetler",
    "summary_tr": "İlk otomasyonunu küçük bir deneyle kur. Lisans, altyapı maliyeti ve güvenli yayın arasındaki farkı baştan gör.",
    "word_count": 740,
    "estimated_duration_sec": 317,
    "narration_text": "n8n’i yerelde öğrenmek: ücretsiz lisans, gerçek maliyetler. İlk otomasyonunu küçük bir deneyle kur. Lisans, altyapı maliyeti ve güvenli yayın arasındaki farkı baştan gör.. n8n’i öğrenmek için ilk hedefin internete açık bir sunucu olmamalı. Önce kendi bilgisayarında, örnek verilerle tek bir işi çalıştırmak daha anlaşılır bir başlangıçtır. Bu rehber kişisel öğrenme içindir; üretim kurulumu veya n8n Assistant altyapısının tamamını kapsamaz. Ücretsiz olan tam olarak ne? Community sürümünün lisans bedeli ile bir otomasyonun toplam maliyeti aynı şey değildir. n8n’in Sustainable Use License koşulları kullanım biçimine bağlıdır. Kendi öğrenme deneyini, başkalarının akış oluşturabildiği bir hizmete dönüştürürken lisans koşullarını yeniden kontrol et. Sınırsız, koşulsuz açık kaynak kullanım hakkı varsayma. Bilgisayarın, yedekleme, barındırma ve dış API çağrıları ayrıca kaynak tüketir. Ücretsiz bir model kotası kullanman da bütün sistemin kalıcı olarak ücretsiz olacağı anlamına gelmez. İlk deneyi küçült Güncel resmi Docker Compose belgesini aç ve hangi bileşenleri kurduğunu tek tek anla. Tam kurulum örneğini tanımadığın yetkilerle kopyalamak yerine, eğitim ortamının ihtiyaçlarını belirle. Deneme verisini gerçek müşteri verisinden ayır. Sürümünü kaydet; yükseltmeden önce kalıcı verinin geri yüklenebilen bir yedeğini al. İlk akış için manuel tetikleyici ve sabit bir metin yeterli olabilir. Sonucu ekranda gör, sonra tek bir dönüşüm adımı ekle. E-posta göndermek, dosya silmek veya hesap ayarlarını değiştirmek bu ilk deneyin parçası olmasın. Ne zaman durup yeniden tasarlamalısın? Akış dışarıdan istek alacaksa TLS, kimlik doğrulama, erişim sınırları ve yedek geri yükleme testi ayrı bir iş paketidir. Bir portu açmış olmak güvenli yayın yaptığın anlamına gelmez. Aşağıdaki listeyi tamamlayıp bir sonraki rehberde insan onaylı bilgi akışına geçebilirsin. Bir akışı anlamak için küçük bir çalışma masası Örnek olarak üç başlıktan oluşan bir okuma listesi düşün. Başlıklar gerçek bir hesaptan gelmesin; onları kendin yaz. Akışın görevi her başlığa bir kategori eklemek ve sonucu liste halinde göstermek olsun. Böylece ağ bağlantısı, yetki veya ücretli çağrı sorunu olmadan veri dönüşümünü gözleyebilirsin. İlk sonuçta hangi alanın kaybolduğunu, hangisinin yeniden adlandırıldığını ve listenin kaç öğeden oluştuğunu kontrol et. Sonraki denemede boş bir başlık ekle. Beklentini akışı çalıştırmadan yaz: Boş değer atlanacak mı, uyarı mı üretilecek, yoksa işlem duracak mı? Bu üç davranışın hepsi belirli bir tasarımda anlamlı olabilir. Sorun, karar verilmeden sistemin rastgele davranmasıdır. Normal örnek kadar eksik veriyi de kaydetmek, ileride akışa yeni düğüm eklediğinde neyin bozulduğunu anlamanı kolaylaştırır. Kurulum notu, ekran görüntüsünden daha değerlidir Kullandığın n8n sürümünü, kurulum belgesinin bağlantısını, kalıcı verinin yerini ve akışı başlatma yöntemini kısa bir notta tut. Bu notta parola, erişim anahtarı veya gerçek kişilerin verileri bulunmasın. Çalışan bir ekran görüntüsü yalnızca o anı gösterir; aynı ortamın yeniden kurulabileceğini kanıtlamaz. Haftalar sonra döndüğünde hangi parçanın sana ait olduğunu bilmek, yeni bir kurulum videosu aramaktan daha faydalı olur. Bir güncelleme öncesinde mevcut akışın örnek çıktısını sakla. Güncellemeden sonra aynı girdiyi verip alanları karşılaştır. Arayüzde bir düğmenin yeri değişmiş olsa bile asıl sorunun veri davranışı olduğunu unutma. Fark varsa önce sürüm notunu ve ilgili düğümün belgesini incele; aynı anda hem kurulum biçimini hem akış mantığını değiştirme. Ne zaman başarılı sayılır? Başarı ölçütün, akışın bir kez yeşil görünmesi değil; aynı örneği yeniden çalıştırabildiğini açıklayabilmen olsun. Hangi verinin nereden geldiğini, nerede dönüştüğünü ve hata durumunda nerede durduğunu tek paragrafta yaz. Sonucu yalnızca sen anlıyorsan notuna bir örnek girdi ve beklenen çıktı ekle. Böyle bir kayıt, kişisel öğrenme deneyini küçük ama sürdürülebilir bir sisteme dönüştürür. Alttaki etkileşimli kontrol listesi bu hazırlığı toparlamak içindir. İşaretlediğin maddeler gerçek ortamını denetlemez ve bilgisayarına komut göndermez. Kendi notlarınla birlikte kullan; eksik kalan bir maddeyi yeni bir araç satın alma gerekçesi yerine öğrenilecek bir sonraki konu olarak ele al. On dakikalık bir yeniden üretim provası Akışı bitirdiğinde bilgisayarın başından kısa süre uzaklaş ve kendi notunu yeni görüyormuş gibi oku. Başlatma adımını, örnek girdiyi ve beklenen çıktıyı bulabiliyor musun? Bir alanın adını hatırlamak zorundaysan onu nota ekle. Burada amaç hız yarışı yapmak değil, deneyin yalnızca kısa süreli hafızana bağlı kalmadığını görmektir. Ekrandaki düğümlerin tamamını anlatmak yerine verinin izlediği yolu açıklamak daha kullanışlı olabilir. İkinci provada akışın başına fazladan bir örnek kayıt koy. Sonucun kaç öğe taşıması gerektiğini önceden say. Ardından bir alanı kaldır ve hata mesajının hangi düğümde belirdiğini not et. Bu iki küçük değişiklik, giriş verisiyle sonuç arasındaki ilişkiyi görünür yapar. Hata durumunda geri dönmek için ilk örneği sakla; çalışan kopyanın üzerine kontrolsüz biçimde yazma. Son olarak notunun paylaşılabilir bir sürümünü çıkar ve bütün sırların, gerçek adreslerin ve kişisel kayıtların dışarıda kaldığını kontrol et. Kaynaklar - Güncel Docker Compose kurulumu - n8n lisansı ve kullanım senaryoları Sıradaki adım Bilgi toplayan bir AI akışı: insan onayı nerede olmalı? rehberiyle devam edebilir veya bu yazının çalışma notlarını PDF olarak indirebilirsin."
  },
  {
    "index": 16,
    "slug": "kucuk-araclar",
    "slug_tr": "kucuk-araclar",
    "title_tr": "Küçük araçlar, büyük kolaylıklar",
    "summary_tr": "Günlük iş akışımı kolaylaştıran küçük web araçları ve arkalarındaki düşünce süreci.",
    "word_count": 81,
    "estimated_duration_sec": 34,
    "narration_text": "Küçük araçlar, büyük kolaylıklar. Günlük iş akışımı kolaylaştıran küçük web araçları ve arkalarındaki düşünce süreci.. Bir fikirle başlıyor Bir ihtiyacın olması yeterli. Bazen tek bir sayfa, bazen küçük bir zamanlayıcı. İyi araçlar gereksiz ayrıntıları azaltır. Bu site de böyle bir deney: düşünmek, üretmek ve öğrendiklerimi paylaşmak için küçük bir alan. Önce küçük bir parça - Gerçek bir ihtiyacı seç. - Kullanılabilir en küçük hâlini kur. - Biraz kullan, sonra yeniden düşün. Bu yazı yerel sürüm için örnek içeriktir; yönetim panelinden düzenlenebilir."
  },
  {
    "index": 17,
    "slug": "yapay-zekayla-uretmek",
    "slug_tr": "yapay-zekayla-uretmek",
    "title_tr": "Yapay zekâyla birlikte üretmek",
    "summary_tr": "Yapay zekâyı bir araçtan öte, yaratıcı bir ortak olarak görmek üzerine kişisel denemeler.",
    "word_count": 58,
    "estimated_duration_sec": 24,
    "narration_text": "Yapay zekâyla birlikte üretmek. Yapay zekâyı bir araçtan öte, yaratıcı bir ortak olarak görmek üzerine kişisel denemeler.. Daha iyi sorular Bir fikri anlatmaya çalışırken onu daha iyi anlıyorum. Yapay zekâyla çalışmak da çoğu zaman böyle: düşünceyi görünür kılmak, alternatifleri denemek ve sonucu dikkatle kontrol etmek. Bu sitedeki sohbet alanı da bu meraktan doğdu. Yerel sürüm için örnek yazı."
  },
  {
    "index": 18,
    "slug": "odaklanmak-icin",
    "slug_tr": "odaklanmak-icin",
    "title_tr": "Daha iyi odaklanmak için neler işe yarıyor?",
    "summary_tr": "Odaklanma rutinlerimi, denediğim yöntemleri ve küçük ama etkili ipuçlarını paylaşıyorum.",
    "word_count": 68,
    "estimated_duration_sec": 29,
    "narration_text": "Daha iyi odaklanmak için neler işe yarıyor?. Odaklanma rutinlerimi, denediğim yöntemleri ve küçük ama etkili ipuçlarını paylaşıyorum.. Küçük bir başlangıç Bir işi bitirmek için her zaman uzun bir zamana ihtiyaç yok. Bazen tek bir işe ayırdığım yirmi beş dakika yeterli. Sitedeki odak zamanlayıcısı bu küçük ritmi denemek için hazır. - Tek bir iş seç. - Bildirimleri bir süreliğine kapat. - Mola vermeyi unutma. Yerel sürüm için örnek içerik."
  }
]

if kac_yazi == "İlk 3 Test Blogu":
    secilen_yazilar = BLOG_CORPUS[:3]
elif kac_yazi == "İlk 5 Blog":
    secilen_yazilar = BLOG_CORPUS[:5]
else:
    secilen_yazilar = BLOG_CORPUS

print(f"✅ {len(secilen_yazilar)} blog yazısı seslendirilmek üzere seçildi:")
for b in secilen_yazilar:
    print(f"  • [{b['index']}/18] {b['slug']} ({b['word_count']} kelime) -> {b['title_tr']}")

---
## 🚀 3. Adım: F5-TTS ile Toplu Seslendirmeyi Başlatın

Seçilen blog yazıları sırayla sentezlenir. Her blog için `output_audios/<slug>-tr.wav` dosyası üretilir.

In [ ]:
#@title 🚀 3. Adım: Toplu Sentezlemeyi Çalıştırın
#@markdown Sentezleme GPU üzerinde başlatılır ve canlı ilerleme gösterilir:

import os
import re
import time
import torch
import soundfile as sf
import numpy as np
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

# 1. Model Kurulumu (Yalnızca GPU üzerinde çalışır)
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ GPU (CUDA) bulunamadı! Lütfen Çalışma Zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU seçin.")

device = "cuda"

if 'f5tts' not in globals():
    print(f"⏳ F5-TTS Türkçe modeli GPU belleğine alınıyor ({torch.cuda.get_device_name(0)})...")
    ckpt_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="f5_tts_turkish_1000000.safetensors")
    vocab_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="vocab.txt")
    
    f5tts = F5TTS(
        model='F5TTS_Base',
        ckpt_file=ckpt_path,
        vocab_file=vocab_path,
        device=device
    )
    print(f"✅ Model hazır! Cihaz: {torch.cuda.get_device_name(0)} ({device.upper()})")

# 2. Fonetik Sözlük
fonetik_sozluk = {
    "React": "Riyekt",
    "HTML": "He-Te-Me-Le",
    "JavaScript": "Cavasikript",
    "DOM": "Dom",
    "CSS": "Si-Es-Es",
    "Next.js": "Nekst ceyes",
    "API": "Eypiyay",
    "JSON": "Ceysın",
    "Suspense": "Saspens",
    "fallback": "folbek",
    "transition": "trenzışın",
    "effect": "efekt",
    "render": "rendır",
    "URL": "Yu-Ar-El",
    "AI": "Ey-Ay",
    "LLM": "El-El-Em",
    "TTS": "Ti-Ti-Es",
    "n8n": "En-sekiz-en",
    "WordPress": "Vordpres"
}

out_dir = "output_audios"
os.makedirs(out_dir, exist_ok=True)
ref_audio = "temiz_referans.wav"
adim_sayisi = kalite_adim_sayisi if 'kalite_adim_sayisi' in globals() else 32

toplam_baslangic = time.time()
uretilen_dosyalar = []

print(f"\n🎙️ Toplam {len(secilen_yazilar)} blog yazısı GPU üzerinde seslendiriliyor (NFE adımı: {adim_sayisi})...\n")

for b_idx, blog in enumerate(secilen_yazilar, 1):
    slug = blog['slug']
    title = blog['title_tr']
    raw_text = blog['narration_text']
    
    # Fonetik uygulama
    t = raw_text
    for k, v in fonetik_sozluk.items():
        t = re.sub(rf'\b{re.escape(k)}\b', v, t, flags=re.IGNORECASE)
        
    cumleler = [c.strip() for c in re.split(r'(?<=[.!?])\s+', t.replace('\n', ' ')) if len(c.strip()) > 3]
    print(f"\n▶️ [{b_idx}/{len(secilen_yazilar)}] Başlatıldı: {slug} ({len(cumleler)} cümle)")
    
    blog_t0 = time.time()
    ses_parcalari = []
    sr_val = 24000
    
    for c_idx, c in enumerate(cumleler, 1):
        c_t0 = time.time()
        with torch.inference_mode():
            wav, sr, _ = f5tts.infer(
                ref_file=ref_audio,
                ref_text=referans_metni,
                gen_text=c,
                nfe_step=adim_sayisi,
                cfg_strength=sadakat_cfg,
                speed=konusma_hizi,
                seed=42,
                remove_silence=False
            )
        sr_val = sr
        ses_parcalari.append(wav)
        if c_idx < len(cumleler):
            ses_parcalari.append(np.zeros(int(0.25 * sr)))
        if c_idx % 5 == 0 or c_idx == len(cumleler):
            print(f"   İlerleme: {c_idx}/{len(cumleler)} cümle ({time.time()-c_t0:.1f} sn/cümle)")
            
    final_wav = np.concatenate(ses_parcalari)
    out_file = os.path.join(out_dir, f"{slug}-tr.wav")
    sf.write(out_file, final_wav, sr_val)
    
    sure = len(final_wav) / sr_val
    uretilen_dosyalar.append(out_file)
    print(f"  ✅ Tamamlandı: {out_file} ({sure:.1f} sn ses | {time.time()-blog_t0:.1f} sn işlem)")

gecen_toplam = time.time() - toplam_baslangic
print(f"\n🎉 TÜM SESLER ÜRETİLDİ! Toplam Süre: {gecen_toplam/60:.1f} dakika ({len(uretilen_dosyalar)} dosya).")


---
## 📦 4. Adım: Tüm Sesleri Zip Olarak İndirin

Üretilen tüm `.wav` dosyaları otomatik olarak `beydahsaglam_blog_sesleri_18.zip` arşivine dönüştürülür ve tarayıcınız üzerinden indirilir.

In [ ]:
#@title 📦 4. Adım: Zip Arşivle ve İndir
#@markdown Tek tıkla tüm sesleri içeren zip dosyasını indirin:

import zipfile
from google.colab import files

zip_name = "beydahsaglam_blog_sesleri_18.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, fs in os.walk(out_dir):
        for f in fs:
            if f.endswith('.wav') or f.endswith('.webm') or f.endswith('.mp3'):
                zipf.write(os.path.join(root, f), arcname=f)

print(f"📦 Zip arşivi hazır: {zip_name} ({os.path.getsize(zip_name)/(1024*1024):.1f} MB)")
print("📥 İndirme başlatılıyor...")
files.download(zip_name)
print("✅ İndirme tamamlandı!")